# 模块概述

WtBtCore 是 WonderTrader 回测框架的核心模块，负责提供各种策略类型的回测模拟环境。主要包括：
- 历史数据回放和管理
- 多种策略类型的回测模拟器（CTA、选股、高频、UFT、执行器）
- 订单撮合引擎
- 事件通知机制
- 辅助工具类

1. **数据层**（HisDataMgr + HisDataReplayer）：
   - 负责历史数据的加载、缓存和回放
   - 支持多种数据类型和回放模式
   - 是整个框架的数据基础

2. **策略层**（各种Mocker）：
   - 提供不同策略类型的回测模拟环境
   - 每个Mocker都实现相应的策略上下文接口和数据接收接口
   - 负责策略执行、持仓管理、盈亏计算等

3. **辅助层**（MatchEngine + EventNotifier）：
   - MatchEngine提供订单撮合功能
   - EventNotifier提供事件通知功能

4. **工具层**（WtHelper）：
   - 提供通用的辅助功能
   - 所有类都可以使用


# 层次关系图
```mermaid
graph TB
    %% 数据层
    subgraph 数据层["数据层 - 历史数据管理"]
        HisDataMgr["HisDataMgr<br/>历史数据管理器<br/>- 动态加载数据读取器模块<br/>- 提供统一的数据加载接口<br/>- 支持K线/Tick/订单队列等数据"]
        HisDataReplayer["HisDataReplayer<br/>历史数据回放器<br/>- 数据缓存管理<br/>- 按K线/Tick/定时任务回放<br/>- 数据订阅和查询<br/>- 复权处理"]
        
        HisDataMgr -->|"使用"| HisDataReplayer
    end
    
    %% 策略模拟器层
    subgraph 策略层["策略层 - 回测模拟器"]
        CtaMocker["CtaMocker<br/>CTA策略回测模拟器<br/>- 继承ICtaStraCtx和IDataSink<br/>- 支持条件单、限价单、止损单<br/>- 持仓管理和盈亏计算<br/>- 图表数据输出"]
        
        SelMocker["SelMocker<br/>选股策略回测模拟器<br/>- 继承ISelStraCtx和IDataSink<br/>- 信号延迟执行机制<br/>- 多明细持仓管理<br/>- T+1规则支持"]
        
        HftMocker["HftMocker<br/>高频交易策略回测模拟器<br/>- 继承IHftStraCtx和IDataSink<br/>- 支持Tick/订单队列/订单明细/逐笔成交<br/>- 订单队列机制<br/>- 异步回测模式"]
        
        UftMocker["UftMocker<br/>UFT极速策略回测模拟器<br/>- 继承IUftStraCtx和IDataSink<br/>- 订单队列机制<br/>- 多空双向持仓<br/>- T+1规则支持"]
        
        ExecMocker["ExecMocker<br/>执行器模拟器<br/>- 继承ExecuteContext/IDataSink/IMatchSink<br/>- 通过撮合引擎执行订单<br/>- 支持多种数量模式<br/>- 订单执行日志"]
    end
    
    %% 辅助组件层
    subgraph 辅助层["辅助层 - 核心组件"]
        MatchEngine["MatchEngine<br/>撮合引擎<br/>- 订单管理<br/>- 限价订单簿维护<br/>- 订单撮合逻辑<br/>- 撤单处理"]
        
        EventNotifier["EventNotifier<br/>事件通知器<br/>- 消息队列服务<br/>- 回测事件通知<br/>- 数据推送<br/>- 资金信息推送"]
    end
    
    %% 工具层
    subgraph 工具层["工具层 - 通用工具"]
        WtHelper["WtHelper<br/>辅助工具类<br/>- 路径管理<br/>- 目录创建<br/>- 跨平台支持<br/>- 静态工具方法"]
    end
    
    %% 接口层（虚拟）
    subgraph 接口层["接口层 - 抽象接口"]
        IDataSink["IDataSink<br/>数据接收接口<br/>- handle_tick<br/>- handle_bar_close<br/>- handle_schedule<br/>- handle_init等"]
        
        ICtaStraCtx["ICtaStraCtx<br/>CTA策略上下文接口"]
        ISelStraCtx["ISelStraCtx<br/>选股策略上下文接口"]
        IHftStraCtx["IHftStraCtx<br/>高频策略上下文接口"]
        IUftStraCtx["IUftStraCtx<br/>UFT策略上下文接口"]
        ExecuteContext["ExecuteContext<br/>执行器上下文接口"]
        IMatchSink["IMatchSink<br/>撮合回调接口"]
    end
    
    %% 数据层关系
    HisDataReplayer -->|"实现"| IDataSink
    
    %% 策略层关系
    CtaMocker -->|"实现"| ICtaStraCtx
    CtaMocker -->|"实现"| IDataSink
    CtaMocker -->|"使用"| HisDataReplayer
    CtaMocker -.->|"可选使用"| EventNotifier
    
    SelMocker -->|"实现"| ISelStraCtx
    SelMocker -->|"实现"| IDataSink
    SelMocker -->|"使用"| HisDataReplayer
    
    HftMocker -->|"实现"| IHftStraCtx
    HftMocker -->|"实现"| IDataSink
    HftMocker -->|"使用"| HisDataReplayer
    
    UftMocker -->|"实现"| IUftStraCtx
    UftMocker -->|"实现"| IDataSink
    UftMocker -->|"使用"| HisDataReplayer
    
    ExecMocker -->|"实现"| ExecuteContext
    ExecMocker -->|"实现"| IDataSink
    ExecMocker -->|"实现"| IMatchSink
    ExecMocker -->|"使用"| HisDataReplayer
    ExecMocker -->|"使用"| MatchEngine
    
    %% 辅助层关系
    MatchEngine -->|"回调"| IMatchSink
    HisDataReplayer -.->|"可选使用"| EventNotifier
    
    %% 工具层关系（所有类都使用）
    CtaMocker -.->|"使用"| WtHelper
    SelMocker -.->|"使用"| WtHelper
    HftMocker -.->|"使用"| WtHelper
    UftMocker -.->|"使用"| WtHelper
    ExecMocker -.->|"使用"| WtHelper
    HisDataReplayer -.->|"使用"| WtHelper
    HisDataMgr -.->|"使用"| WtHelper
    EventNotifier -.->|"使用"| WtHelper
    MatchEngine -.->|"使用"| WtHelper
    
    %% 样式定义
    classDef dataLayer fill:#e1f5ff,stroke:#01579b,stroke-width:2px
    classDef strategyLayer fill:#f3e5f5,stroke:#4a148c,stroke-width:2px
    classDef helperLayer fill:#fff3e0,stroke:#e65100,stroke-width:2px
    classDef toolLayer fill:#e8f5e9,stroke:#1b5e20,stroke-width:2px
    classDef interfaceLayer fill:#fce4ec,stroke:#880e4f,stroke-width:2px,stroke-dasharray: 5 5
    
    class HisDataMgr,HisDataReplayer dataLayer
    class CtaMocker,SelMocker,HftMocker,UftMocker,ExecMocker strategyLayer
    class MatchEngine,EventNotifier helperLayer
    class WtHelper toolLayer
    class IDataSink,ICtaStraCtx,ISelStraCtx,IHftStraCtx,IUftStraCtx,ExecuteContext,IMatchSink interfaceLayer
```


# 数据层

## HisDataMgr.h/cpp - 历史数据管理器
管理历史数据的加载和读取，通过动态库加载数据读取器模块

```cpp
class HisDataMgr : public IBtDtReaderSink
```

参考 [Includes/note.ipynb/数据管理接口层/回测数据读取 IBtDtReader.h/回测数据读取回调接口类 IBtDtReaderSink](../Includes/note.ipynb)

### 成员
- `IBtDtReader* _reader`：数据读取器指针

### IBtDtReaderSink 接口实现

#### 数据读取器日志回调 reader_log
```cpp
/**
 * @brief 数据读取器日志回调函数实现
 * @param ll 日志级别
 * @param message 日志消息
 */
void HisDataMgr::reader_log(WTSLogLevel ll, const char* message)
{
	WTSLogger::log_raw(ll, message); // 将日志转发到系统日志
}
```

### 初始化 init
从参数 cfg 中读取对应 "module" 的字段来加载动态库，通过动态库中 "createBtDtReader" 函数创建数据读取器 `_reader`，并且 `_reader` 初始化 init
```cpp
/**
 * @brief 初始化历史数据管理器
 * @param cfg 配置信息（包含模块路径等）
 * @return 是否初始化成功
 */
bool HisDataMgr::init(WTSVariant* cfg)
```

### 数据加载

#### 加载原始K线数据 load_raw_bars
调用 `_reader` 的 read_raw_bars 方法读取数据，并且调用回调函数 cb 返回数据
```cpp
/**
 * @brief 加载原始K线数据
 * @param exchg 交易所代码
 * @param code 合约代码
 * @param period K线周期
 * @param cb 数据加载回调函数
 * @return 是否加载成功
 */
bool HisDataMgr::load_raw_bars(const char* exchg, const char* code, WTSKlinePeriod period, FuncLoadDataCallback cb)
```

#### 加载原始Tick数据 load_raw_ticks
调用 `_reader` 的 read_raw_ticks 方法读取数据，并且调用回调函数 cb 返回数据
```cpp
/**
 * @brief 加载原始Tick数据
 * @param exchg 交易所代码
 * @param code 合约代码
 * @param uDate 日期
 * @param cb 数据加载回调函数
 * @return 是否加载成功
 */
bool HisDataMgr::load_raw_ticks(const char* exchg, const char* code, uint32_t uDate, FuncLoadDataCallback cb)
```

#### 加载原始订单队列数据 load_raw_ordque
调用 `_reader` 的 read_raw_order_queues 方法读取数据，并且调用回调函数 cb 返回数据
```cpp
/**
 * @brief 加载原始订单队列数据
 * @param exchg 交易所代码
 * @param code 合约代码
 * @param uDate 日期
 * @param cb 数据加载回调函数
 * @return 是否加载成功
 */
bool HisDataMgr::load_raw_ordque(const char* exchg, const char* code, uint32_t uDate, FuncLoadDataCallback cb)
```

#### 加载原始订单明细数据 load_raw_orddtl
调用 `_reader` 的 read_raw_order_details 方法读取数据，并且调用回调函数 cb 返回数据
```cpp
/**
 * @brief 加载原始订单明细数据
 * @param exchg 交易所代码
 * @param code 合约代码
 * @param uDate 日期
 * @param cb 数据加载回调函数
 * @return 是否加载成功
 */
bool HisDataMgr::load_raw_orddtl(const char* exchg, const char* code, uint32_t uDate, FuncLoadDataCallback cb)
```

#### 加载原始逐笔成交数据 load_raw_trans
调用 `_reader` 的 read_raw_transactions 方法读取数据，并且调用回调函数 cb 返回数据
```cpp
/**
 * @brief 加载原始逐笔成交数据
 * @param exchg 交易所代码
 * @param code 合约代码
 * @param uDate 日期
 * @param cb 数据加载回调函数
 * @return 是否加载成功
 */
bool HisDataMgr::load_raw_trans(const char* exchg, const char* code, uint32_t uDate, FuncLoadDataCallback cb)
```

## HisDataReplayer.h/cpp - 历史数据回放器
历史数据的回放和模拟，是整个回测框架的数据核心

### 架构图
```mermaid
graph TB
    %% 接口层
    IDataSink["IDataSink<br/>数据接收器接口<br/>接收回放数据"]
    IBtDataLoader["IBtDataLoader<br/>数据加载器接口<br/>可选外部数据源"]
    
    %% 主类
    HisDataReplayer["HisDataReplayer<br/>历史数据回放器<br/>数据加载/缓存/回放核心"]
    
    %% 外部依赖
    HisDataMgr["HisDataMgr<br/>数据管理器"]
    
    %% 策略模拟器
    Mocker["策略模拟器<br/>CtaMocker/SelMocker等<br/>实现IDataSink"]
    
    %% 核心关系
    HisDataReplayer -->|"注册回调"| IDataSink
    HisDataReplayer -.->|"可选"| IBtDataLoader
    HisDataReplayer -->|"使用"| HisDataMgr
    HisDataReplayer -.->|"推送数据"| Mocker
    
    Mocker -.->|"实现"| IDataSink
    Mocker -->|"使用"| HisDataReplayer
    
    %% 样式
    classDef interface fill:#fce4ec,stroke:#880e4f,stroke-width:2px,stroke-dasharray:5 5
    classDef main fill:#e1f5ff,stroke:#01579b,stroke-width:3px
    classDef internal fill:#fff3e0,stroke:#e65100,stroke-width:2px
    classDef cache fill:#f3e5f5,stroke:#4a148c,stroke-width:2px
    classDef external fill:#e0f2f1,stroke:#004d40,stroke-width:2px
    classDef mocker fill:#f1f8e9,stroke:#33691e,stroke-width:2px
    
    class IDataSink,IBtDataLoader interface
    class HisDataReplayer main
    class TickCache,BarsCache cache
    class Mocker mocker
```

### 数据接收器接口 IDataSink
接口类，用于接收历史数据回放器 HisDataReplayer 推送的各种市场数据，由 `策略层/*Mocker` 实现
```cpp
class IDataSink
```

#### 数据接收回调

##### Tick数据回调 handle_tick
```cpp
/**
 * @brief 处理Tick数据回调
 * 
 * @param stdCode 合约代码
 * @param curTick 当前Tick数据
 * @param pxType 价格类型
 */
virtual void	handle_tick(const char* stdCode, WTSTickData* curTick, uint32_t pxType) = 0;  // 处理Tick数据（纯虚函数）
```

##### 订单队列数据回调 handle_order_queue
```cpp
/**
 * @brief 处理订单队列数据回调
 * 
 * @param stdCode 合约代码
 * @param curOrdQue 当前订单队列数据
 */
virtual void	handle_order_queue(const char* stdCode, WTSOrdQueData* curOrdQue) {};  // 处理订单队列数据（默认空实现）
```

##### 订单明细数据回调 handle_order_detail
```cpp
/**
 * @brief 处理订单明细数据回调
 * 
 * @param stdCode 合约代码
 * @param curOrdDtl 当前订单明细数据
 */
virtual void	handle_order_detail(const char* stdCode, WTSOrdDtlData* curOrdDtl) {};  // 处理订单明细数据（默认空实现）
```

##### 逐笔成交数据回调 handle_transaction
```cpp
/**
 * @brief 处理逐笔成交数据回调
 * 
 * @param stdCode 合约代码
 * @param curTrans 当前逐笔成交数据
 */
virtual void	handle_transaction(const char* stdCode, WTSTransData* curTrans) {};  // 处理逐笔成交数据（默认空实现）
```

##### K线收盘回调 handle_bar_close
```cpp
/**
 * @brief 处理K线收盘回调
 * 
 * @param stdCode 合约代码
 * @param period 周期
 * @param times 倍数
 * @param newBar 新的K线数据
 */
virtual void	handle_bar_close(const char* stdCode, const char* period, uint32_t times, WTSBarStruct* newBar) = 0;  // 处理K线收盘（纯虚函数）
```

#### 生命周期与事件回调

##### 初始化回调 handle_init
```cpp
/**
 * @brief 处理初始化回调
 */
virtual void	handle_init() = 0; // 处理初始化（纯虚函数）
```

##### 交易时段开始回调 handle_session_begin
```cpp
/**
 * @brief 处理交易时段结束回调
 * 
 * @param curTDate 当前交易日期
 */
virtual void	handle_session_end(uint32_t curTDate) = 0; // 处理交易时段结束（纯虚函数）
```

##### 交易时段结束回调 handle_session_end
```cpp
/**
 * @brief 处理交易时段结束回调
 * 
 * @param curTDate 当前交易日期
 */
virtual void	handle_session_end(uint32_t curTDate) = 0; // 处理交易时段结束（纯虚函数）
```

##### 定时调度回调 handle_schedule
```cpp
/**
 * @brief 处理定时调度回调
 * 
 * @param uDate 日期
 * @param uTime 时间
 */
virtual void	handle_schedule(uint32_t uDate, uint32_t uTime) = 0;  // 处理定时调度（纯虚函数）
```

##### 回放完成回调 handle_replay_done
```cpp
/**
 * @brief 处理回放完成回调
 */
virtual void	handle_replay_done() {} // 处理回放完成（默认空实现）
```

##### 小节结束回调 handle_section_end
```cpp
/**
 * @brief 处理小节结束回调
 * 
 * @param curTDate 当前交易日期
 * @param curTime 当前时间
 */
virtual void	handle_section_end(uint32_t curTDate, uint32_t curTime) {}  // 处理小节结束（默认空实现）
```

### 回测数据加载器接口 IBtDataLoader
用于从外部数据源加载历史数据
```cpp
class IBtDataLoader
```

#### 方法

##### 加载最终历史K线数据 loadFinalHisBars
```cpp
/**
 * @brief 加载最终历史K线数据
 * 
 * 和loadRawHisBars的区别在于，loadFinalHisBars系统认为是最终所需数据，不再进行加工，
 * 例如复权数据、主力合约数据等。loadRawHisBars是加载未加工的原始数据的接口。
 *
 * @param obj 回传用的，原样返回即可
 * @param stdCode 合约代码
 * @param period K线周期
 * @param cb 回调函数
 * @return 是否加载成功
 */
virtual bool loadFinalHisBars(void* obj, const char* stdCode, WTSKlinePeriod period, FuncReadBars cb) = 0;  // 加载最终历史K线数据
```

##### 加载原始历史K线数据 loadRawHisBars
```cpp
/**
 * @brief 加载原始历史K线数据
 *
 * @param obj 回传用的，原样返回即可
 * @param stdCode 合约代码
 * @param period K线周期
 * @param cb 回调函数
 * @return 是否加载成功
 */
virtual bool loadRawHisBars(void* obj, const char* stdCode, WTSKlinePeriod period, FuncReadBars cb) = 0;  // 加载原始历史K线数据
```

##### 加载全部除权因子 loadAllAdjFactors
```cpp
/**
 * @brief 加载全部除权因子
 * 
 * @param obj 回传用的，原样返回即可
 * @param cb 回调函数
 * @return 是否加载成功
 */
virtual bool loadAllAdjFactors(void* obj, FuncReadFactors cb) = 0;  // 加载全部除权因子
```

##### 根据合约加载除权因子 loadAdjFactors
```cpp
/**
 * @brief 根据合约加载除权因子
 *
 * @param obj 回传用的，原样返回即可
 * @param stdCode 合约代码
 * @param cb 回调函数
 * @return 是否加载成功
 */
virtual bool loadAdjFactors(void* obj, const char* stdCode, FuncReadFactors cb) = 0;  // 根据合约加载除权因子
```

##### 加载历史Tick数据 loadRawHisTicks
```cpp
/**
 * @brief 加载历史Tick数据
 * 
 * @param obj 回传用的，原样返回即可
 * @param stdCode 合约代码
 * @param uDate 日期
 * @param cb 回调函数
 * @return 是否加载成功
 */
virtual bool loadRawHisTicks(void* obj, const char* stdCode, uint32_t uDate, FuncReadTicks cb) = 0;  // 加载历史Tick数据
```

##### 是否自动转储为dsb格式 isAutoTrans
```cpp
/**
 * @brief 是否自动转储为dsb格式
 * @return 是否自动转储
 */
virtual bool isAutoTrans() { return true; } // 是否自动转储为dsb（默认返回true）
```

### 历史数据回放器类 HisDataReplayer
负责历史数据的加载、缓存和回放，支持多种回放模式
```cpp
class HisDataReplayer
```

#### 成员

- **核心管理器指针与基础配置**
  - `IDataSink* _listener`：指向数据接收器（策略模拟器），回放时通过该指针推送数据
  - `IBtDataLoader* _bt_loader`：指向回测数据加载器（外部数据源），从外部加载历史数据（K线、Tick、复权因子等）
  - `std::string _stra_name`：策略名称，标识当前回测的策略
  - `std::string _base_dir`：基础数据目录路径，定位历史数据文件（二进制、CSV等）
  - `std::string _mode`：回放模式（"bars"/"tasks"/"ticks"），决定回放方式
  - `WTSBaseDataMgr _bd_mgr`：基础数据管理器，查询合约信息（`WTSCommodityInfo`）和交易时段（`WTSSessionInfo`）
  - `WTSHotMgr _hot_mgr`：主力合约管理器，解析连续合约代码（如 "SHFE.rb.HOT" → "SHFE.rb2410"）
  - `HisDataMgr _his_dt_mgr`：历史数据管理器，加载历史数据（K线、Tick、订单队列等）
  - `EventNotifier* _notifier`：事件通知器指针（可选），发送回测事件和数据通知
- **K线数据缓存**
  - `BarsCache _bars_cache`：已订阅的K线缓存。使用游标跟踪回放进度
  - `BarsCache _unbars_cache`：存储未订阅但需要的K线数据。
    - typedef wt_hashmap<std::string, `BarsListPtr`> BarsCache
      ```cpp
      typedef struct _BarsList
      {
        std::string   _code;          // 合约代码
        WTSKlinePeriod _period;       // K线周期
        uint32_t      _cursor;        // 游标（已回放的K线条数）
        uint32_t      _count;         // 总K线条数
        uint32_t      _times;         // 倍数（如m5表示5分钟）
        std::vector<WTSBarStruct> _bars;  // K线数据列表
        double        _factor;        // 最后一条复权因子
        uint32_t      _untouch_days;  // 未用到的天数（用于缓存清理）
      } BarsList;
      ```

- **高频数据缓存**
  - `TickCache _ticks_cache`：Tick数据缓存。
  - `OrdDtlCache _orddtl_cache`：订单明细缓存。
  - `OrdQueCache _ordque_cache`：订单队列缓存。
  - `TransCache _trans_cache`：逐笔成交缓存。
    - typedef wt_hashmap<std::string, `HftDataList<WTSTickStruct>`> TickCache
    - typedef wt_hashmap<std::string, `HftDataList<WTSOrdDtlStruct>`> OrdDtlCache
    - typedef wt_hashmap<std::string, `HftDataList<WTSOrdQueStruct>`> OrdQueCache
    - typedef wt_hashmap<std::string, `HftDataList<WTSTransStruct>`> TransCache
      ```cpp
      template <typename T>
      class HftDataList
      {
        std::string   _code;          // 合约代码
        uint32_t      _date;          // 日期
        std::size_t   _cursor;        // 游标（已回放的数据条数）
        std::size_t   _count;         // 总数据条数
        std::vector<T> _items;        // 数据项列表
      };
      ```

- **订阅管理**
  - `wt_hashset<std::string> _codes_in_subbed`：已订阅的合约代码集合。
  - `wt_hashset<std::string> _codes_in_unsubbed`：未订阅的合约代码集合。
  - `wt_hashset<std::string> _unsubbed_in_need`：未订阅但需要的K线集合，支持不订阅而直接指定合约下单的场景
  - `StraSubMap _tick_sub_map`：Tick数据订阅表。
  - `StraSubMap _ordque_sub_map`：订单队列数据订阅表。
  - `StraSubMap _orddtl_sub_map`：订单明细数据订阅表。
  - `StraSubMap _trans_sub_map`：逐笔成交数据订阅表。
    - typedef wt_hashmap<std::string, `SubList`> StraSubMap：合约代码 -> 订阅列表
    - typedef wt_hashmap<uint32_t, `SubOpt`> SubList：上下文ID -> 订阅选项
    - typedef std::pair<uint32_t, uint32_t> SubOpt：上下文ID，订阅选项：0-原始，1-前复权，2-后复权
- **时间状态管理**
  - `uint32_t _cur_date`：当前日期（YYYYMMDD）。
  - `uint32_t _cur_time`：当前时间（HHMM）。
  - `uint32_t _cur_secs`：当前秒数（0-86399）。
  - `uint32_t _cur_tdate`：当前交易日期（YYYYMMDD）。
  - `uint32_t _closed_tdate`：已关闭的交易日期，用于交易日转换和T+1规则处理
  - `uint32_t _opened_tdate`：已打开的交易日期，用于交易日转换和T+1规则处理
  - `uint64_t _begin_time`：开始时间（时间戳）。
  - `uint64_t _end_time`：结束时间（时间戳）。
- **回放配置与状态标志**
  - `std::string _main_key`：主键（主合约代码周期），标识主合约和主周期
  - `std::string _min_period`：最小K线周期
  - `std::string _main_period`：主周期。
  - `bool _tick_enabled`：是否开启了Tick回测。
  - `bool _tick_simulated`：是否需要模拟Tick数据。
  - `bool _align_by_section`：重采样分钟线是否按小节对齐。
  - `bool _nosim_if_notrade`：如果没有成交量则不模拟Tick。
  - `bool _running`：是否正在运行。
  - `bool _terminated`：是否已终止。

- **定时任务管理**
  - `TaskInfoPtr _task`：定时任务信息指针，支持按定时任务回放模式
    - typedef std::shared_ptr<`TaskInfo`> TaskInfoPtr
      ```cpp
      typedef struct _TaskInfo
      {
        uint32_t        _id;            // 任务ID
        char            _name[16];      // 任务名
        char            _trdtpl[16];    // 交易日模板
        char            _session[16];   // 交易时间模板
        uint32_t        _day;           // 日期（根据周期变化）
        uint32_t        _time;          // 时间（精确到分钟）
        bool            _strict_time;   // 是否是严格时间
        uint64_t        _last_exe_time; // 上次执行时间
        TaskPeriodType  _period;        // 任务周期
      } TaskInfo;
      ```

- **手续费与价格映射**
  - `FeeMap _fee_map`：手续费映射表，存储各合约的手续费配置
    - typedef wt_hashmap<std::string, `FeeItem`> FeeMap
      ```cpp
      typedef struct _FeeItem
      {
        double  _open;          // 开仓手续费
        double  _close;         // 平仓手续费
        double  _close_today;   // 平今手续费
        bool    _by_volume;     // 是否按手数收费
      } FeeItem;
      ```
  - `PriceMap _price_map`：价格映射表，存储各合约的最新价格
    - typedef `wt_hashmap<std::string, double>` PriceMap

- **复权因子管理**
  - `AdjFactorMap _adj_factors`：复权因子映射表，存储各合约的复权因子列表
    - typedef wt_hashmap<std::string, `AdjFactorList`> AdjFactorMap
    - typedef std::vector<`AdjFactor`> AdjFactorList
      ```cpp
      typedef struct _AdjFactor
      {
        uint32_t  _date;        // 日期
        double    _factor;      // 复权因子
      } AdjFactor;
      ```
  - `uint32_t _adjust_flag`：复权标记（位运算），记录复权选项（1-成交量复权，2-成交额复权，4-总持复权）

- **Tick模拟辅助缓存**
  - `std::map<std::string, WTSTickStruct> _day_cache`：每日Tick缓存。当Tick回放未开放时，缓存每日的Tick数据
  - `std::map<std::string, std::string> _ticker_keys`：专门服务于 Tick模拟 (simTicks) 功能
    - 当在策略中对同一个合约订阅了多个周期的 K 线时，回测引擎不需要对这三个周期都模拟一遍 Tick，只需要用精度最高的那一个周期来模拟即可
    - 键：标准合约代码（例如 "SHFE.rb.2310"） 
    - 值：`_bars_cache`（K线缓存）中，该合约 *精度最高（周期最小）* 的那一份K线数据的索引键，"{合约代码}#{周期标识}#{倍数}"

- **缓存清理配置**
  - `uint32_t _cache_clear_days`：缓存自动清理天数，设置缓存自动清理的天数阈值

#### 初始化与配置

##### 初始化历史数据回放器 init

##### 准备回放（加载数据缓存等）prepare
- 如果正在运行即 `_running` 为 true，返回 false
- 设置 `_running`、`_terminated`，调用 reset() 进行重置
- 设置 `_cur_date`、`_cur_time`、`_cur_secs`、`_cur_tdate`
- 事件通知器 `_notifier` 调用 notifyEvent("BT_START") 通知开始回测
- 调用数据接收器 `_listener` 的初始化回调 handle_init
- 如果未启用Tick回测即 `_tick_enabled` 为 false，调用 checkUnbars
```cpp
/**
 * @brief 准备回放
 * 初始化回放环境，重置状态，设置初始时间
 * @return 是否准备成功
 */
bool HisDataReplayer::prepare()
```

##### 设置回放时间范围 set_time_range
```cpp
/**
 * @brief 设置回放时间范围
 * @param stime 开始时间
 * @param etime 结束时间
 */
inline void set_time_range(uint64_t stime, uint64_t etime)
{
    _begin_time = stime;
    _end_time = etime;
}
```

##### 启用/禁用Tick回放 enable_tick
```cpp
/**
 * @brief 启用/禁用Tick回放
 * @param bEnabled 是否启用（默认true）
 */
inline void enable_tick(bool bEnabled = true)
{
    _tick_enabled = bEnabled;
}
```

##### 加载手续费配置 loadFees
从文件 filename 中加载手续费配置到 `_fee_map`
```cpp
/**
 * @brief 加载手续费配置
 * 从JSON格式的手续费配置文件中加载所有合约的手续费信息
 * @param filename 手续费配置文件路径
 */
void HisDataReplayer::loadFees(const char* filename)
```

##### 重置回放器状态 reset

#### 回放控制方法

##### 运行回测 run

##### 停止回测 stop

##### 按照K线进行回测 run_by_bars
用于 CTA 策略（K线驱动）回测：以主K线为时间步长有序推进回测进度，负责管理交易时段的开收盘状态、通过 插值模拟（simTicks） 还原盘中价格波动，并精确触发K线收盘与定时任务回调

过程：
- 获取主K：barsList = `_bars_cache`[`_main_key`]，根据 `_begin_time` 和 `_end_time` 获取回测范围 sIdx 和 eIdx
- **当未终止!_terminated时一直循环**：
  - 如果 barsList 的游标没初始化：跳出循环
  - 否则：
    - 计算下一条K线时间 nextDate 和 nextTime，如果其超出了_end_time：跳出循环
    - **如果日线，或分钟线但nextTime没到交易时段收盘时间**
      - 计算下一个交易日期 nextTDate
      - **如果交易日期 _opened_tdate 与 nextTDate 不一致（换日了）**
        - 如果关闭日期 `_closed_tdate` 和 `_opened_tdate` 不一致（昨天还没收盘）
          - 调用引擎 `_listener` 的交易时段结束回调 handle_session_end(_cur_tdate)
          - 关闭日期 `_closed_tdate` 置为 _cur_date
          - 清空 `_day_cache`
        - 获取交易时段开盘时间，并使用其更新 `_cur_date`、`_cur_time`、`_cur_secs`（0）
        - 调用引擎 `_listener` 的交易时段开始回调 handle_session_begin(nextTDate)
        - 调用 check_cache_days 来检查缓存天数并清理过期缓存
        - `_opened_tdate` 和 `_cur_tdate` 都置为 nextTDate
    - **如果开启了Tick回放_tick_enabled**
      - 调用 replayHftDatas：将 _cur_date * 10000 + _cur_time ~ nextBarTime 的所有订阅合约的4类高频数据，每次各从中挑选出时间最早的那一个事件给引擎
      - 置 `_tick_simulated` 为 replayHftDatas 是否调用成功
    - 否则
      - 调用 checkUnbars 检查未订阅的K线缓存
    - 更新 `_cur_date` = nextDate、`_cur_time` = nextTime、`_cur_secs` = 0
    - 判断交易日是否结束 isEndTDate = _cur_time 到了收盘时间
    - 如果需要模拟Tick `_tick_simulated`
      - 调用 simTicks(_cur_date, _cur_time, _cur_tdate) 将 `_bars_cache` 中的K线数据拆解为 4 笔虚拟的 Tick 数据（开、高、低、收），推送给策略，模拟价格波动
    - 如果未启用Tick回放 !`_tick_enabled`
      - 调用 simTickWithUnsubBars(_cur_date, _cur_time, _cur_tdate) 将 `_unbars_cache` 中的K线数据拆解为 4 笔虚拟的 Tick 数据（开、高、低、收），推送给策略，模拟价格波动
    - **调用 onMinuteEnd 来处理 `_bars_cache` 中过期K线**
    - 如果交易小节结束：
      - 调用引擎 `_listener` 的小节结束回调 handle_section_end(nextDate, nextTime)
    - 如果交易日结束 && 还未关闭_closed_tdate != _cur_tdate
      - 调用引擎 `_listener` 的交易时段结束回调 handle_session_end(_cur_tdate)
      - 关闭日期 `_closed_tdate` 置为 _cur_date
      - 清空 `_day_cache`
- 调用事件通知器 `_notifier` 的回测结束回调 notifyEvent("BT_END")
- 如果最后一个交易日还未关闭 _closed_tdate != _cur_tdate
  - 调用引擎 `_listener` 的交易时段结束回调 handle_session_end(_cur_tdate)
- 调用引擎 `_listener` 的调用回放完成回调 handle_replay_done
```cpp
/**
 * @brief 按照K线进行回测
 * 按照主K线周期逐条回放K线数据，并在每条K线收盘时触发回调
 * @param bNeedDump 是否将回测进度落地到文件中（默认false）
 */
void HisDataReplayer::run_by_bars(bool bNeedDump /* = false */)
```

##### 按照Tick进行回测 run_by_ticks
用于纯高频回测模式：以交易日为单位循环推进回测进度，完成当日高频数据（订单明细—>逐笔成交—>Tick—>订单队列）的全量回放
- **while((`_cur_tdate` <= `_end_time` 对应的结束交易日期) && 没有终止!_terminated)**
  - 如果：`_ticks_cache` 中至少有一个对应 `_tick_sub_map` 中合约，且日期为 _cur_tdate 的Tick缓存
    - 调用引擎 `_listener` 的交易时段开始回调 handle_session_begin
    - 调用 check_cache_days 来检查缓存天数并清理过期缓存
    - 调用 replayHftDatasByDay：将 _cur_tdate 的订阅合约的 订单明细—>逐笔成交—>Tick—>订单队列，循环每次各从中挑选出时间最早的数据给引擎 `_listener`
    - 调用引擎 `_listener` 的交易时段结束回调 handle_session_end(_cur_tdate)
  - _cur_tdate 置为其下一天
- **调用引擎 `_listener` 的回放完成回调 handle_replay_done**
- **调用事件通知器 `_notifier` 的回测结束回调 notifyEvent("BT_END")**
```cpp
/**
 * @brief 按照Tick进行回测
 * @param bNeedDump 是否将回测进度落地到文件中（默认false）
 */
void HisDataReplayer::run_by_ticks(bool bNeedDump /* = false */)
```

##### 按照定时任务进行回测 run_by_tasks
通常回测有两种驱动方式
- **数据驱动**：每一根K线或每一个Tick到来时触发一次逻辑
- **时间/任务驱动**：无论有没有特定的行情数据，按照固定的时间间隔唤醒策略执行逻辑

该函数是驱动基于 **时间调度任务** 的回测模式
- 一个死循环（直到回测结束），根据用户注册的 `_task`（任务信息），控制回测时间的推进
- 当时间推进到任务设定的触发点时，通过 `_listener`->handle_schedule 通知策略层进行操作。

根据 _task->_period 有两个分支：
- 分支一（非分钟级）：是跳跃式的。按天翻日历，只在关键时刻醒来
- 分支二（分钟级）：是流淌式的。按分钟走时间，模拟日内的价格波动流

| 特性 | 分支一 (非分钟级) | 分支二 (分钟级) |
| --- | --- | --- |
| **Tick模拟** | **无** (通常不模拟 Tick) | **有** (调用 `simTicks` 4次) |
| **K线处理** | 只有在任务触发的那一刻，才会去 Check 是否有 K 线闭合。 | 每一小步（如5分钟）都会 Check K 线闭合。 |
| **中间过程** | **黑盒**。中间的波动完全被忽略，只看触发时刻的状态。 | **白盒**。尽可能还原日内的价格路径。 |
| **性能** | **极快**。回测10年可能只需要几秒（因为只跑了几千次循环）。 | **较慢**。回测10年可能需要几分钟（因为跑了几十万次循环）。 |

过程：
- **如果是非分钟级任务**（根据 `_task`->_period 判断）
  - 计算任务触发时间 endtime 为 _task->_time 的前一分钟；如果跨日了，调整当前日期 _cur_date 为其自己的前一天
  - **未终止时（_terminated）一直循环**：
    - **如果当前时间 _cur_time 等于任务触发时间 endtime（准备检查日期是否符合触发条件）**：
      - 如果当前交易日 _cur_tdate 的前一天为节假日：一直找当前面的第一个交易日
      - **根据任务周期 _task->_period 判断是否应该真正触发**：
        - 如果日级任务：应该触发
        - 如果月级任务：如果是下述情况之一
          - 当前日期 _cur_date 和任务触发日期 _task->_day 一致
          - 如果当前交易日和前一个交易日之间有节假日，任务触发日期 _task->_day 在这两个交易日之间
        - 如果周级任务：如果是下述情况之一
          - 当前日期 _cur_date 和任务触发日期 _task->_day 对应的星期一致
          - 如果当前交易日和前一个交易日之间有节假日，任务触发日期 _task->_day 在这两个交易日之间
        - 如果是年级任务：
          - 如果当前交易日和前一个交易日之间有节假日，任务触发日期 _task->_day 在这两个交易日之间
    - **如果不触发**
      - **更新 _cur_time**，如果当前时间 _cur_time 小于任务触发时间 endtime：
        - 将 _cur_time 置为 endtime，**进入下一轮循环**
      - **更新 _cur_tdate**，使用基础数据管理器 `_bd_mgr` 计算新的交易日期，如果新的交易日期和当前交易日期 _cur_tdate 不一致
        - 将 _cur_tdate 置为计算出来的新的交易日期 newTDate
        - 调用引擎 `_listener` 的 handle_session_begin(newTDate) 来进行交易时段开始回调
        - 调用 check_cache_days 来检查缓存天数并清理过期缓存
        - 调用引擎 `_listener` 的 handle_session_end(newTDate) 来进行交易时段结束回调
    - **否则（触发）**
      - 调用引擎 `_listener` 的 handle_session_begin(_cur_tdate) 来进行交易时段开始回调
      - 调用 check_cache_days 来检查缓存天数并清理过期缓存
      - 调用 onMinuteEnd(curDate, curTime, _cur_tdate/前一个交易日) 来处理 `_bars_cache` 中过期K线
        - 其中是 _cur_tdate 还是前一个交易日，取决于 endtime 是否到了交易日结束时间
      - 调用引擎 `_listener` 的 handle_session_end(newTDate) 来进行交易时段结束回调
    - _cur_date 更新到其下一天
    - _cur_time 设为任务触发时间 endtime
    - _cur_tdate，由 `_bd_mgr` 用更新后的 _cur_date、_cur_time 来计算新的交易日期生成
    - 如果 _cur_date * 10000 + _cur_time 超过了结束时间 _end_time
      - 调用引擎 `_listener` 的交易时段结束回调 handle_session_end(_cur_tdate)、回放完成回调 handle_replay_done
      - 调用事件通知器 `_notifier` 的回测结束回调 notifyEvent("BT_END")
      - **跳出循环**
- **否则**
  - 调用引擎 `_listener` 的交易时段开始回调 handle_session_begin(_cur_tdate)
  - 调用 check_cache_days 来检查缓存天数并清理过期缓存
  - **未终止时（_terminated）一直循环**：
    - 计算 mins = _cur_time 在交易时段内的分钟数
    - **如果 mins < 交易日总分钟数**
      - 调用 simTicks(_cur_date, _cur_time, _cur_tdate) 将 `_bars_cache` 中的K线数据拆解为 4 笔虚拟的 Tick 数据（开、高、低、收），推送给策略，模拟价格波动
      - 调用 onMinuteEnd(_cur_date, _cur_time, 0) 来处理 `_bars_cache` 中过期K线
    - **否则（到了新的交易日）**
      - 标记 bNewTDate 为 true
      - _cur_time 置为交易日收盘时间
      - 调用 simTicks(_cur_date, _cur_time, _cur_tdate) 将 `_bars_cache` 中的K线数据拆解为 4 笔虚拟的 Tick 数据（开、高、低、收），推送给策略，模拟价格波动
      - 调用 onMinuteEnd(_cur_date, _cur_time, _cur_tdate) 来处理 `_bars_cache` 中过期K线
    - **如果是新交易日（根据 bNewTDate）**
      - mins = _task->_time
      - _cur_date = _cur_tdate (如果交易时段偏移分钟数 > 0)， _cur_tdate 的下一个交易日 (如果交易时段偏移分钟数 < 0)
      - _cur_tdate = 其下一个交易日
      - _cur_time = mins 对应的事件
      - 调用引擎 `_listener` 的 handle_session_begin(_cur_tdate) 来进行交易时段开始回调
      - 调用 check_cache_days 来检查缓存天数并清理过期缓存
    - **否则**
      - mins += _task->_time，如果 mins 超过了总交易分钟数：mins 置为总交易分钟数
      - newTime = mins 对应的时间
      - 如果 mins 跨日了：将 _cur_date 置为其自己的下一天
      - 计算当前时间 _cur_time 的分钟数 dayMins，下一个时间 newTime 的分钟数 nextDMins
      - 是否到达新小节 bNewSec = (nextDMins - dayMins > _task->_time) 并且 (没有跨日)
      - 一直循环：当 bNewSec && _cur_date 是节假日
        - _cur_date 置为其下一天
      - _cur_time 置为 newTime
    - 计算下一个任务触发时间 nextTime = _cur_date * 10000 + _cur_time
    - 如果 nextTime > _end_time
      - 调用引擎 `_listener` 的交易时段结束回调 handle_session_end(_cur_tdate)、回放完成回调 handle_replay_done
      - 调用事件通知器 `_notifier` 的回测结束回调 notifyEvent("BT_END")
      - **跳出循环**
```cpp
/**
 * @brief 按照定时任务进行回测
 * 时间调度任务不为空，则按照时间调度任务回放
 * 支持分钟、每日、每周、每月、每年等不同周期的定时任务
 * @param bNeedDump 是否将回测进度落地到文件中（默认false）
 */
void HisDataReplayer::run_by_tasks(bool bNeedDump /* = false */)
```

#### 数据订阅方法

##### 注册数据接收器 register_sink

##### 注册定时任务 register_task

##### 订阅Tick数据 sub_tick

##### 订阅订单队列数据 sub_order_queue

##### 订阅订单明细数据 sub_transaction

##### 订阅逐笔成交数据 sub_transaction

#### 数据查询方法

##### 获取K线切片 get_kline_slice

##### 获取Tick切片 get_tick_slice

##### 获取订单明细切片 get_order_detail_slice

##### 获取订单队列切片 get_order_queue_slice

##### 获取逐笔成交切片 get_transaction_slice

##### 获取最新Tick数据 get_last_tick

#### 时间与状态查询方法

##### 获取当前日期 get_date

##### 获取当前分钟时间 get_min_time

##### 获取当前原始时间 get_raw_time

##### 获取当前秒数 get_secs

##### 获取当前交易日期 get_trading_date

#### 价格与合约信息查询方法

##### 获取当前价格 get_cur_price

##### 获取日价格（开盘价、最高价、最低价、收盘价）get_day_price

##### 获取交易时段信息 get_session_info

##### 获取合约信息 get_commodity_info

##### 获取原始合约代码 get_rawcode

#### 手续费计算方法

##### 计算手续费 calc_fee

#### 缓存管理方法

##### 清空所有缓存 clear_cache

##### 检查缓存天数并清理过期缓存 check_cache_days
- 如果缓存清理天数 `_cache_clear_days` 为 0：返回
- 遍历K线缓存集合 `_bars_cache`：
  - 如果当前缓存项是主K线（键等于 `_main_key`）：跳过（主K线数据不清理）
  - 获取对应的K线列表指针 `barsList`，将其未使用天数计数器 `_untouch_days` 加 1
  - 如果 `_untouch_days` >= 缓存清理天数 `_cache_clear_days`：
    - 从 `_bars_cache` 中删除对应的缓存项
```cpp
/**
 * @brief 检查并清理过期缓存
 * * 根据配置的缓存清理天数，清理长时间未使用的K线缓存
 */
void HisDataReplayer::check_cache_days()
```

##### 检查是否已缓存Tick数据 checkTicks
检查指定合约的Tick数据是否已缓存到指定日期，如果未缓存则尝试加载
- 如果 `_ticks_cache` 中没有对应合约 stdCode 的Tick数据，或者有但是日期不是 uDate：则需要从缓存中加载
  - 注意：如果有对应合约，且日期符合，但是数据条数为 0，则返回 false
- 如果需要从缓存中加载：
  - 如果回测数据加载器 `_bt_loader` 存在：调用 cacheRawTicksFromLoader 进行加载
  - 否则：
    - 如果加载模式 `_mode` 为 "csv"：调用 cacheRawTicksFromCSV 从 csv 文件中加载
    - 否则：调用 cacheRawTicksFromBin 从二进制文件中加载
- 如果没加载到Tick数据：`_ticks_cache`[stdCode] 置空，返回 false
```cpp
/**
 * @brief 检查Tick数据是否已缓存
 * @param stdCode 合约代码
 * @param uDate 交易日期
 * @return 是否成功缓存或已经缓存
 */
bool HisDataReplayer::checkTicks(const char* stdCode, uint32_t uDate)
```

##### 检查是否已缓存订单明细数据 checkOrderDetails

##### 检查是否已缓存订单队列数据 checkOrderQueues

##### 检查是否已缓存逐笔成交数据 checkTransactions

##### 检查所有Tick数据是否已缓存 checkAllTicks`
检查 `_ticks_cache` 中是否至少有一个对应 `_tick_sub_map` 中合约，且日期为 uDate 的Tick缓存
```cpp
/**
 * @brief 检查所有Tick数据是否已缓存
 * @param uDate 交易日期
 * @return 是否至少有一个合约的Tick数据已缓存
 */
bool HisDataReplayer::checkAllTicks(uint32_t uDate)
{
	bool bHasTick = false;
	for (auto& v : _tick_sub_map)
	{
		bHasTick = checkTicks(v.first.c_str(), uDate) || bHasTick;
	}
	return bHasTick;
}
```

##### 检查未订阅的K线缓存 checkUnbars
- 遍历未订阅但需要的合约代码集合 `_unsubbed_in_need`：
  - 如果已在未订阅K线集合 `_codes_in_unsubbed` 或已订阅K线集合 `_codes_in_subbed` 中：跳过
  - 加载数据到 `_unbars_cache`：
    - 如果外部数据加载器 `_bt_loader` 存在，调用 `cacheFinalBarsFromLoader()` 加载
    - 否则：
      - 如果模式 `_mode ` 为 "csv"，调用 `cacheRawBarsFromCSV()` 从CSV文件缓存K线数据
      - 否则调用 `cacheRawBarsFromBin()` 从二进制文件缓存K线数据
    - 如果上述过程没有加载到数据：跳过
  - 将合约代码添加到未订阅K线集合 `_codes_in_unsubbed`
  - 初始化 `_unbars_cache[对应合约]` 的游标 _cursor：将游标定位到当前回测时间对应的K线位置

```cpp
/**
 * @brief 检查未订阅的K线缓存
 * 对于未订阅但需要的合约，自动加载主K线周期的数据并定位到当前时间
 */
void HisDataReplayer::checkUnbars()
```

#### 数据回放与模拟方法

##### 回放HFT数据 replayHftDatas
将 stime ~ etime 的所有订阅合约的高频数据（订单明细、逐笔成交、Tick、订单队列），每次各从中挑选出时间最早的那一个事件给引擎

过程：
- 当未终止时一直循环（!_terminated）
  - 获取各类高频数据的最早未播放时间 nextTime
  - 更新引擎时间：使用 nextTime 更新 `_cur_date`、`_cur_time`、`_cur_secs`
  - 按照 订单明细 ——> 逐笔成交 ——> Tick ——> 订单队列 顺序，访问 ：
    - 也就是分别访问 `_orddtl_sub_map`、`_trans_sub_map`、`_tick_sub_map`、`_ordque_sub_map` 中的游标处数据
    - 如果游标时间不大于 nextTime
    - 分别调用引擎 `_listener` 的 handle_order_detail/handle_transaction/handle_tick/handle_order_queue 回调
- 最后返回的是回调函数总调用次数
```cpp
/**
 * @brief 回放HFT数据
 * 
 * 在指定的时间范围内，按照时间顺序回放HFT数据（Tick、逐笔成交、订单明细、订单队列）
 * 
 * @param stime 开始时间
 * @param etime 结束时间
 * @return 是否成功回放（如果找到数据返回true，否则返回false）
 */
bool HisDataReplayer::replayHftDatas(uint64_t stime, uint64_t etime)
```

##### 按天回放HFT数据 replayHftDatasByDay
将指定日期 (curDate) 的所有订阅合约的高频数据（订单明细、逐笔成交、Tick、订单队列），每次各从中挑选出时间最早的那一个事件给引擎

过程：
- 当未终止时一直循环（!_terminated）
  - 获取各类高频数据的最早未播放时间 nextTime
  - 更新引擎时间：使用 nextTime 更新 `_cur_date`、`_cur_time`、`_cur_secs`
  - 按照 订单明细 ——> 逐笔成交 ——> Tick ——> 订单队列 顺序，访问 ：
    - 也就是分别访问 `_orddtl_sub_map`、`_trans_sub_map`、`_tick_sub_map`、`_ordque_sub_map` 中的游标处数据
    - 如果游标时间不大于 nextTime
    - 分别调用引擎 `_listener` 的 handle_order_detail/handle_transaction/handle_tick/handle_order_queue 回调
- 最后返回的是回调函数总调用次数
```cpp
/**
 * @brief 按天回放HFT数据
 * @param curTDate 当前交易日期
 * @return 回放的Tick总数
 */
uint64_t HisDataReplayer::replayHftDatasByDay(uint32_t curTDate)
```

##### 模拟Tick数据 simTicks
将 `_bars_cache` 中的K线数据拆解为 Tick 数据（根据参数选择开盘价/最高价/最低价/收盘价），推送给策略，模拟价格波动
- 实际中会被调用 4 次，即将一根 K 线拆解为 4 笔虚拟的 Tick 数据（开、高、低、收）

过程：
- 当前时间 nowTime = uDate * 10000 + uTime
- **遍历K线缓存 `_bars_cache`**
  - **如果非日线 && 对应游标 _cursor 没有指向最后，循环处理**：
    - **如果当前游标指向K线时间等于nowTime && (没有禁止0成交模拟!_nosim_if_notrade || K线成交量不为0)**
      - **如果当前指向K线的精度和是 `_ticker_keys` 中存储的对应合约的精度**
        - 获取每日Tcik缓存 WTSTickStruct& curTS = `_day_cache`[对应合约]
        - 使用 _cur_date、_cur_time、成交量vol 更新其 action_date、action_time、volume、total_volume
        - 根据参数 pxType，选择当前K线的 open/high/low/close 设置其 price
        - 更新 
            - open = (open == 0) ? price : open
            - high = max(price, high)
            - low = (low == 0) ? price : min(price, low)
        - 更新 `_price_map`[对应合约] = price
        - **调用引擎 `_listener` 的Tick数据回调 handle_tick(对应合约, curTs, 价格类型pxType)**
      - 处理 `_bars_cache` 中的下一个缓存
    - **否则如果当前游标指向K线时间 < nowTime**
      - 游标移动
    - **否则**
      - 处理 `_bars_cache` 中的下一个缓存
  - **如果日线 && 对应游标 _cursor 没有指向最后，循环处理**
    - **如果当前游标指向K线的日期等于endTDate**
      - **如果当前指向K线的精度和是 `_ticker_keys` 中存储的对应合约的精度**
        - 创建Tick数据 WTSTickStruct curTS 
        - 获取对应合约的交易日收盘时间 curTime
        - 使用 _cur_date、curTime、成交量vol 更新其 action_date、action_time、volume
        - 根据参数 pxType，选择当前K线的 open/high/low/close 设置其 price
        - 更新 `_price_map`[对应合约] = price
        - **调用引擎 `_listener` 的Tick数据回调 handle_tick(对应合约, curTs, 价格类型pxType)**
      - 处理 `_bars_cache` 中的下一个缓存
    - **否则如果当前游标指向K线时间 < nowTime**
      - 游标移动
    - **否则**
      - 处理 `_bars_cache` 中的下一个缓存

```cpp
/**
 * @brief 模拟Tick数据
 * @param uDate 日期
 * @param uTime 时间
 * @param endTDate 结束交易日期（默认0，表示不限制）
 * @param pxType 价格类型（0-开盘价，1-最高价，2-最低价，3-收盘价，默认0）
 */
void HisDataReplayer::simTicks(uint32_t uDate, uint32_t uTime, uint32_t endTDate /* = 0 */, int pxType /* = 0 */)
```

##### 使用未订阅的K线模拟Tick数据 simTickWithUnsubBars
将 `_unbars_cache` 中的K线数据拆解为 Tick 数据（根据参数选择开盘价/最高价/最低价/收盘价），推送给策略，模拟价格波动
- 实际中会被调用 4 次，即将一根 K 线拆解为 4 笔虚拟的 Tick 数据（开、高、低、收）

过程：
- 当前时间 nowTime = uDate * 10000 + uTime
- **遍历K线缓存 `_unbars_cache`**
  - **如果非日线 && 对应游标 _cursor 没有指向最后，循环处理**：
    - **如果当前游标指向K线时间等于nowTime && (没有禁止0成交模拟!_nosim_if_notrade || K线成交量不为0)**
      - 获取每日Tcik缓存 WTSTickStruct& curTS = `_day_cache`[对应合约]
      - 使用 _cur_date、_cur_time、成交量vol 更新其 action_date、action_time、volume、total_volume
      - 根据参数 pxType，选择当前K线的 open/high/low/close 设置其 price
      - 更新 
          - open = (open == 0) ? price : open
          - high = max(price, high)
          - low = (low == 0) ? price : min(price, low)
      - 更新 `_price_map`[对应合约] = price
      - **调用引擎 `_listener` 的Tick数据回调 handle_tick(对应合约, curTs, 价格类型pxType)**
      - 处理 `_unbars_cache` 中的下一个缓存
    - **否则如果当前游标指向K线时间 < nowTime**
      - 游标移动
    - **否则**
      - 处理 `_unbars_cache` 中的下一个缓存
  - **如果日线 && 对应游标 _cursor 没有指向最后，循环处理**
    - **如果当前游标指向K线的日期等于endTDate**
      - 创建Tick数据 WTSTickStruct curTS 
      - 获取对应合约的交易日收盘时间 curTime
      - 使用 _cur_date、curTime、成交量vol 更新其 action_date、action_time、volume
      - 根据参数 pxType，选择当前K线的 open/high/low/close 设置其 price
      - 更新 `_price_map`[对应合约] = price
      - **调用引擎 `_listener` 的Tick数据回调 handle_tick(对应合约, curTs, 价格类型pxType)**
      - 处理 `_unbars_cache` 中的下一个缓存
    - **否则如果当前游标指向K线时间 < nowTime**
      - 游标移动
    - **否则**
      - 处理 `_unbars_cache` 中的下一个缓存

```cpp
/**
 * @brief 模拟Tick数据
 * @param uDate 日期
 * @param uTime 时间
 * @param endTDate 结束交易日期（默认0，表示不限制）
 * @param pxType 价格类型（0-开盘价，1-最高价，2-最低价，3-收盘价，默认0）
 */
void HisDataReplayer::simTicks(uint32_t uDate, uint32_t uTime, uint32_t endTDate /* = 0 */, int pxType /* = 0 */)
```

#### 时间查询辅助方法

##### 获取下一个Tick时间 getNextTickTime
在所有订阅的合约（`_tick_sub_map`）中，对于在 `_ticks_cache` 中有日期为 curTDate 的缓存的，找到 stime 后最近的Tick时间（根据游标）。 
- 这里的Tick时间是 WTSTransStruct 的 action_time
```cpp
/**
 * @brief 获取下一个Tick时间
 * @param curTDate 当前交易日期
 * @param stime 开始时间（默认UINT64_MAX，表示不限制）
 * @return 下一个Tick时间戳，如果没有则返回UINT64_MAX
 */
uint64_t HisDataReplayer::getNextTickTime(uint32_t curTDate, uint64_t stime /* = UINT64_MAX */)
```

##### 获取下一个订单队列时间 getNextOrdQueTime
在所有订阅的合约（`_ordque_sub_map`）中，对于在 `_ordque_cache` 中有日期为 curTDate 的缓存的，找到 stime 后最近的订单队列时间（根据游标）。 
- 这里的Tick时间是 WTSOrdQueStruct 的 action_time
```cpp
/**
 * @brief 获取下一个订单队列时间
 * @param curTDate 当前交易日期
 * @param stime 开始时间（默认UINT64_MAX，表示不限制）
 * @return 下一个订单队列时间戳，如果没有则返回UINT64_MAX
 */
uint64_t HisDataReplayer::getNextOrdQueTime(uint32_t curTDate, uint64_t stime /* = UINT64_MAX */)
```

##### 获取下一个订单明细时间 getNextOrdDtlTime
在所有订阅的合约（`_orddtl_sub_map`）中，对于在 `_orddtl_cache` 中有日期为 curTDate 的缓存的，找到 stime 后最近的订单明细时间（根据游标）。 
- 这里的Tick时间是 WTSTransStruct 的 action_time
```cpp
/**
 * @brief 获取下一个订单明细时间
 * @param curTDate 当前交易日期
 * @param stime 开始时间（默认UINT64_MAX，表示不限制）
 * @return 下一个订单明细时间戳，如果没有则返回UINT64_MAX
 */
uint64_t HisDataReplayer::getNextOrdDtlTime(uint32_t curTDate, uint64_t stime /* = UINT64_MAX */)
```

##### 获取下一个逐笔成交时间 getNextTransTime
在所有订阅的合约（`_trans_sub_map`）中，对于在 `_trans_cache` 中有日期为 curTDate 的缓存的，找到 stime 后最近的逐笔成交时间（根据游标）。 
- 这里的Tick时间是 WTSTransStruct 的 action_time
```cpp
/**
 * @brief 获取下一个逐笔成交时间
 * @param curTDate 当前交易日期
 * @param stime 开始时间（默认UINT64_MAX，表示不限制）
 * @return 下一个逐笔成交时间戳，如果没有则返回UINT64_MAX
 */
uint64_t HisDataReplayer::getNextTransTime(uint32_t curTDate, uint64_t stime /* = UINT64_MAX */)
```

#### 复权因子管理方法

##### 从文件加载股票复权因子 loadStkAdjFactorsFromFile

##### 从外部加载器加载股票复权因子 loadStkAdjFactorsFromLoader

#### 内部辅助方法

##### 处理分钟线结束 onMinuteEnd
检查是否有 K 线在这一分钟闭合（完成），如果有，就通知策略进行处理，同时触发定时任务。
- 遍历K线缓存 `_bars_cache`
  - 对于非日线，如果其游标 _cursor 没在最后，循环处理 [游标, uDate * 10000 + uTime] 中的K线：
    - 调用数据接收器 `_listener` 的K线收盘回调 handle_bar_close
  - 对于日线，如果其游标 _cursor 没在最后，循环处理 [游标, endTDate] 之间的K线：
    - 调用数据接收器 `_listener` 的K线收盘回调 handle_bar_close
- 遍历未订阅K线缓存 `_unbars_cache`
  - 对于非日线，如果其游标 _cursor 没在最后，将游标调整到 > uDate * 10000 + uTime
  - 对于日线，如果其游标 _cursor 没在最后，将游标调整到 > endTDate
- 调用数据接收器 `_listener` 的 handle_schedule(uDate, uTime) 来通知引擎
```cpp
/**
 * @brief 处理分钟线结束
 * @param uDate 日期
 * @param uTime 时间
 * @param endTDate 结束交易日期（默认0，表示不限制）
 * @param tickSimulated 是否模拟Tick（默认true）
 */
void HisDataReplayer::onMinuteEnd(uint32_t uDate, uint32_t uTime, uint32_t endTDate /* = 0 */, bool tickSimulated /* = true */)
````

# 策略层

## CtaMocker.h/cpp - CTA策略回测模拟器
模拟CTA（Commodity Trading Advisor）策略在历史数据上的交易执行过程
```cpp
class CtaMocker : public ICtaStraCtx, public IDataSink
```

### 架构图
```mermaid
classDiagram
    class ICtaStraCtx {
        <<interface>>
        CTA策略上下文接口
        为策略提供交易上下文环境
    }
    
    class IDataSink {
        <<interface>>
        数据接收接口
        接收历史数据回放器推送的市场数据
    }
    
    class CtaMocker {
        CTA策略回测模拟器
        模拟CTA策略在历史数据上的交易执行过程
    }
    
    class HisDataReplayer {
        历史数据回放器
        加载和回放历史市场数据
        提供数据查询接口
    }
    
    class EventNotifier {
        事件通知器
        发送回测事件和数据通知
        支持消息队列推送
    }
    
    class CtaStrategy {
        <<interface>>
        CTA策略接口
        用户实现的策略逻辑
    }
    
    class ICtaStrategyFact {
        <<interface>>
        策略工厂接口
        创建和管理策略实例
    }
    
    class StraFactInfo {
        策略工厂信息
        管理动态库加载和策略工厂
    }
    
    ICtaStraCtx <|.. CtaMocker : 实现
    IDataSink <|.. CtaMocker : 实现
    CtaMocker --> HisDataReplayer : 使用
    CtaMocker --> EventNotifier : 使用
    CtaMocker --> CtaStrategy : 管理
    CtaMocker --> StraFactInfo : 包含
    StraFactInfo --> ICtaStrategyFact : 包含
    ICtaStrategyFact ..> CtaStrategy : 创建
```

### 成员
- **核心管理器指针与基础配置**
  - `uint32_t _context_id`：上下文唯一标识符，用于区分不同的策略实例
  - `HisDataReplayer* _replayer`：历史数据回放器指针，用于获取历史数据和市场信息
  - `EventNotifier* _notifier`：事件通知器指针（可选），用于通知回测进度和结果

- **统计信息**
  - `uint64_t _total_calc_time`：策略总计算时间（微秒），用于性能统计
  - `uint32_t _emit_times`：策略总计算次数，用于性能统计
  - `uint32_t _schedule_times`：调度次数，用于标识当前是第几次调度

- **滑点设置**
  - `int32_t _slippage`：成交滑点，如果是比例滑点则为万分比，否则为价格跳动单位
  - `bool _ratio_slippage`：是否为比例滑点模式

- **主K线信息**
  - `std::string _main_key`：主K线键值（格式：合约代码#周期）
  - `std::string _main_code`：主K线合约代码
  - `std::string _main_period`：主K线周期

- **K线状态管理**
  - `KlineTags _kline_tags`：K线标签映射表，用于标记K线的收盘状态和通知状态
    - typedef wt_hashmap<std::string, `KlineTag`> KlineTags
        ```cpp
        typedef struct _KlineTag
        {
        bool  _closed;    // K线是否已收盘
        bool  _notify;     // 是否已通知策略K线收盘
        } KlineTag;
        ```

- **价格缓存**
  - `PriceMap _price_map`：价格映射表，用于缓存上一笔价格
    - typedef `wt_hashmap<std::string, double>` PriceMap

- **持仓明细信息**
  - `PositionMap _pos_map`：持仓映射表，记录每个合约的持仓汇总信息
    - typedef wt_hashmap<std::string, `PosInfo`> PositionMap
        ```cpp
        typedef struct _PosInfo
        {
        double  _volume;          // 总持仓数量（正数表示多仓，负数表示空仓）
        double  _closeprofit;      // 已实现盈亏
        double  _dynprofit;        // 浮动盈亏
        uint64_t _last_entertime;  // 最后开仓时间
        uint64_t _last_exittime;   // 最后平仓时间
        double  _frozen;           // 冻结持仓（T+1规则下，当日开仓的持仓会被冻结）
        std::vector<DetailInfo> _details;  // 持仓明细列表
        } PosInfo;

        typedef struct _DetailInfo
        {
        bool    _long;            // 是否多头持仓
        double  _price;           // 开仓价格
        double  _volume;          // 持仓数量
        uint64_t _opentime;        // 开仓时间（时间戳）
        uint32_t _opentdate;       // 开仓交易日
        double  _max_profit;       // 持仓期间最大盈利
        double  _max_loss;         // 持仓期间最大亏损
        double  _max_price;        // 持仓期间最高价
        double  _min_price;        // 持仓期间最低价
        double  _profit;           // 当前浮动盈亏
        char    _opentag[32];      // 开仓标签
        uint32_t _open_barno;      // 开仓K线编号
        } DetailInfo;
        ```
  - `double _total_closeprofit`：累计已实现盈亏

- **交易信号信息**
  - `SignalMap _sig_map`：信号映射表，记录待执行的交易信号
    - typedef wt_hashmap<std::string, `SigInfo`> SignalMap
        ```cpp
        typedef struct _SigInfo
        {
        double    _volume;      // 目标仓位
        std::string _usertag;    // 用户标签
        double    _sigprice;     // 信号生成时的价格
        double    _desprice;     // 指定成交价格（如果为0则使用市场价格）
        uint32_t  _sigtype;      // 信号类型：0-调度中发出，1-非调度中发出，2-条件单触发
        uint64_t  _gentime;      // 信号生成时间（时间戳）
        } SigInfo;
        ```

- **条件单管理**
  - `CondEntrustMap _condtions`：条件单映射表（合约代码 -> 条件单列表）
    - typedef wt_hashmap<std::string, `CondList`> CondEntrustMap
    - typedef std::vector<`CondEntrust`> CondList
        ```cpp
        typedef struct _CondEntrust
        {
        WTSCompareField _field;    // 比较字段类型（如最新价、开盘价等）
        WTSCompareType  _alg;      // 比较算法类型（等于、大于、小于等）
        double    _target;         // 目标价格，用于条件判断
        double    _qty;            // 委托数量
        char      _action;         // 委托动作：0-开多,1-平多,2-开空,3-平空,4-设置仓位
        char      _code[MAX_INSTRUMENT_LENGTH];  // 合约代码
        char      _usertag[32];    // 用户标签
        } CondEntrust;
        ```

- **日志流**
  - `std::stringstream _trade_logs`：成交日志流，用于记录所有成交记录
  - `std::stringstream _close_logs`：平仓日志流，用于记录所有平仓记录
  - `std::stringstream _fund_logs`：资金日志流，用于记录每日资金曲线
  - `std::stringstream _sig_logs`：信号日志流，用于记录所有交易信号
  - `std::stringstream _pos_logs`：持仓日志流，用于记录每日持仓情况
  - `std::stringstream _index_logs`：指标日志流，用于记录指标数据
  - `std::stringstream _mark_logs`：标记日志流，用于记录图表标记

- **调度状态**
  - `bool _is_in_schedule`：是否在自动调度中，用于标识当前是否在策略计算过程中

- **用户数据**
  - `StringHashMap _user_datas`：用户数据映射表（键 -> 值），用于策略存储自定义数据
    - typedef `wt_hashmap<std::string, std::string>` StringHashMap
  - `bool _ud_modified`：用户数据是否被修改，用于判断是否需要持久化

- **资金信息**
  - `StraFundInfo _fund_info`：策略资金信息实例
    ```cpp
    typedef struct _StraFundInfo
    {
      double  _total_profit;      // 累计已实现盈亏
      double  _total_dynprofit;   // 累计浮动盈亏
      double  _total_fees;         // 累计手续费
    } StraFundInfo;
    ```

- **策略工厂和策略实例**
  - `StraFactInfo _factory`：策略工厂信息实例
    ```cpp
    typedef struct _StraFactInfo
    {
      std::string   _module_path;      // 策略模块路径
      DllHandle     _module_inst;      // 动态库句柄
      ICtaStrategyFact* _fact;         // 策略工厂指针
      FuncCreateStraFact _creator;      // 创建策略工厂的函数指针
      FuncDeleteStraFact _remover;      // 删除策略工厂的函数指针
    } StraFactInfo;
    ```
  - `CtaStrategy* _strategy`：CTA策略实例指针

- **异步回测控制**
  - `StdUniqueMutex _mtx_calc`：计算互斥锁，用于异步回测模式下的线程同步
  - `StdCondVariable _cond_calc`：计算条件变量，用于异步回测模式下的线程等待和通知
  - `bool _has_hook`：是否安装了计算钩子（人为控制是否启用钩子）
  - `bool _hook_valid`：钩子是否有效（根据是否是异步回测模式而确定钩子是否可用）
  - `std::atomic<uint32_t> _cur_step`：当前步骤，用于控制异步回测的状态机（0-初始，1-计算中，2-计算完成，3-计算完成确认）
  - `bool _in_backtest`：是否在回测中
  - `bool _wait_calc`：是否等待计算完成

- **数据持久化**
  - `bool _persist_data`：是否对回测结果持久化

- **时间信息**
  - `uint32_t _cur_tdate`：当前交易日
  - `uint32_t _cur_bartime`：当前K线时间
  - `uint64_t _last_cond_min`：最后设置条件单的时间（分钟）

- **Tick订阅**
  - `wt_hashset<std::string> _tick_subs`：tick订阅列表，记录已订阅tick数据的合约代码

- **图表配置**
  - `std::string _chart_code`：图表K线合约代码
  - `std::string _chart_period`：图表K线周期
  - `std::unordered_map<std::string, ChartIndex> _chart_indice`：图表指标映射表（指标名称 -> 指标信息）
    ```cpp
    typedef struct _ChartIndex
    {
      std::string _name;                // 指标名称
      uint32_t    _indexType;          // 指标类型
      std::unordered_map<std::string, ChartLine> _lines;      // 指标线条映射表
      std::unordered_map<std::string, double> _base_lines;    // 基准线映射表
    } ChartIndex;
    
    typedef struct _ChartLine
    {
      std::string _name;                // 线条名称
      uint32_t    _lineType;            // 线条类型
    } ChartLine;
    ```

### 初始化与配置方法

#### 初始化CTA策略工厂 init_cta_factory

#### 加载增量回测数据 load_incremental_data

#### 安装计算钩子 install_hook

#### 启用或禁用计算钩子 enable_hook

#### 单步计算 step_calc

### IDataSink 接口实现

#### 处理tick数据推送 handle_tick

#### 处理K线收盘事件 handle_bar_close

#### 处理定时调度事件 handle_schedule

#### 处理初始化事件 handle_init

#### 处理交易日开始事件 handle_session_begin

#### 处理交易日结束事件 handle_session_end

#### 处理小节结束事件 handle_section_end

#### 处理回放完成事件 handle_replay_done

### ICtaStraCtx 接口实现

#### 策略生命周期回调

##### 获取上下文ID id

##### 策略初始化回调 on_init

##### 交易日开始回调 on_session_begin

##### 交易日结束回调 on_session_end

##### tick数据回调 on_tick

##### K线数据回调 on_bar

##### 定时调度回调 on_schedule

##### 枚举持仓 enum_position

##### tick数据更新回调 on_tick_updated

##### K线收盘回调 on_bar_close

##### 策略计算回调 on_calculate

#### 交易操作接口

##### 开多仓 stra_enter_long

##### 开空仓 stra_enter_short

##### 平多仓 stra_exit_long

##### 平空仓 stra_exit_short

##### 获取持仓数量 stra_get_position

##### 设置目标仓位 stra_set_position

#### 价格与时间信息接口

##### 获取最新价格 stra_get_price

##### 读取当日价格 stra_get_day_price

##### 获取当前交易日 stra_get_tdate

##### 获取当前日期 stra_get_date

##### 获取当前时间 stra_get_time

#### 资金信息接口

##### 获取资金数据 stra_get_fund_data

#### 持仓信息接口

##### 获取首次开仓时间 stra_get_first_entertime

##### 获取最后开仓时间 stra_get_last_entertim

##### 获取最后平仓时间 stra_get_last_exittime

##### 获取最后开仓价格 stra_get_last_enterprice

##### 获取最后开仓标签 stra_get_last_entertag

##### 获取持仓均价 stra_get_position_avgpx

##### 获取持仓盈亏 stra_get_position_profit

##### 获取指定标签的持仓开仓时间 stra_get_detail_entertime

##### 获取指定标签的持仓成本 stra_get_detail_cost

##### 获取指定标签的持仓盈亏 stra_get_detail_profit

#### 市场数据接口

##### 获取合约信息 stra_get_comminfo

##### 获取K线数据切片 stra_get_bars

##### 获取tick数据切片 stra_get_ticks

##### 获取最新tick数据 stra_get_last_tick

##### 订阅tick数据 stra_sub_ticks

##### 订阅K线收盘事件 stra_sub_bar_events

##### 获取分月合约代码 stra_get_rawcode

#### 日志接口

##### 记录信息日志 stra_log_info

##### 记录调试日志 stra_log_debug

##### 记录警告日志 stra_log_warn

##### 记录错误日志 stra_log_error

#### 用户数据接口

##### 保存用户数据 stra_save_user_data

##### 加载用户数据 stra_load_user_data

#### 图表数据接口

##### 设置图表K线 set_chart_kline

##### 添加图表标记 add_chart_mark

##### 注册指标 register_index

##### 注册指标线 register_index_line

##### 添加指标基准线 add_index_baseline

##### 设置指标值 set_index_value

### 内部辅助方法

#### 输出回测结果到CSV文件 dump_outputs

#### 输出策略数据到JSON文件 dump_stradata

#### 输出图表数据到JSON和CSV文件 dump_chartdata

#### 记录交易信号日志 log_signal

#### 记录成交日志 log_trade

#### 记录平仓日志 log_close

#### 更新持仓的动态盈亏 update_dyn_profit

#### 执行仓位设置操作 do_set_position

#### 添加交易信号 append_signal

#### 获取指定合约的条件单列表 get_cond_entrusts

#### 处理tick数据 proc_tick

## SelMocker.h/cpp - 选股策略回测模拟器
模拟选股策略在历史数据上的执行过程
```cpp
class SelMocker : public ISelStraCtx, public IDataSink
```

### 架构图
```mermaid
classDiagram
    class ISelStraCtx {
        <<interface>>
        选股策略上下文接口
        为策略提供交易上下文环境
    }
    
    class IDataSink {
        <<interface>>
        数据接收接口
        接收历史数据回放器推送的市场数据
    }
    
    class SelMocker {
        选股策略回测模拟器
        模拟选股策略在历史数据上的交易执行过程
    }
    
    class HisDataReplayer {
        历史数据回放器
        加载和回放历史市场数据
        提供数据查询接口
    }
    
    class SelStrategy {
        <<interface>>
        选股策略接口
        用户实现的策略逻辑
    }
    
    class ISelStrategyFact {
        <<interface>>
        策略工厂接口
        创建和管理策略实例
    }
    
    class StraFactInfo {
        策略工厂信息
        管理动态库加载和策略工厂
    }
    
    ISelStraCtx <|.. SelMocker : 实现
    IDataSink <|.. SelMocker : 实现
    SelMocker --> HisDataReplayer : 使用
    SelMocker --> SelStrategy : 管理
    SelMocker --> StraFactInfo : 包含
    StraFactInfo --> ISelStrategyFact : 包含
    ISelStrategyFact ..> SelStrategy : 创建
```

### 成员
- **核心管理器指针与基础配置**
  - `uint32_t _context_id`：上下文唯一标识符，用于区分不同的策略实例
  - `HisDataReplayer* _replayer`：历史数据回放器指针，用于获取历史数据和合约信息

- **统计信息**
  - `uint64_t _total_calc_time`：总计算时间（微秒），用于统计策略执行性能
  - `uint32_t _emit_times`：总计算次数，用于统计策略执行次数
  - `uint32_t _schedule_times`：调度次数，记录定时调度被调用的次数

- **滑点设置**
  - `int32_t _slippage`：成交滑点，单位取决于_ratio_slippage
  - `bool _ratio_slippage`：是否比例滑点，true表示比例滑点（单位：万分之一），false表示绝对滑点（单位：最小变动价位）

- **主键信息**
  - `std::string _main_key`：主键字符串，用于标识策略实例

- **K线状态管理**
  - `KlineTags _kline_tags`：K线标签映射表，用于跟踪每个合约每个周期的K线状态
    - typedef `wt_hashmap<std::string, KlineTag>` KlineTags
    ```cpp
    typedef struct _KlineTag
    {
      bool    _closed;    // 是否已闭合
      uint32_t _count;    // 闭合次数，记录该K线周期被闭合的次数
    } KlineTag;
    ```

- **价格缓存**
  - `PriceMap _price_map`：价格映射表，缓存每个合约的最新价格和时间戳
    - typedef `wt_hashmap<std::string, PriceInfo>` PriceMap
    - typedef `std::pair<double, uint64_t>` PriceInfo（价格和时间戳）

- **持仓明细信息**
  - `PositionMap _pos_map`：持仓映射表，记录每个合约的持仓信息
    - typedef `wt_hashmap<std::string, PosInfo>` PositionMap
    ```cpp
    typedef struct _PosInfo
    {
      double    _volume;          // 总持仓数量（正数为做多，负数为做空）
      double    _closeprofit;     // 已平仓盈亏金额
      double    _dynprofit;        // 动态盈亏金额
      uint64_t  _last_entertime;  // 最后一次开仓时间（纳秒时间戳）
      uint64_t  _last_exittime;   // 最后一次平仓时间（纳秒时间戳）
      double    _frozen;           // 冻结持仓数量（T+1规则下使用）
      std::vector<DetailInfo> _details;  // 持仓明细列表
    } PosInfo;
    
    typedef struct _DetailInfo
    {
      bool    _long;            // 是否做多
      double  _price;           // 开仓价格
      double  _volume;          // 持仓数量
      uint64_t _opentime;        // 开仓时间（纳秒时间戳）
      uint32_t _opentdate;       // 开仓交易日日期（格式：YYYYMMDD）
      double  _max_profit;       // 最大盈利金额
      double  _max_loss;         // 最大亏损金额
      double  _max_price;        // 最高价格
      double  _min_price;        // 最低价格
      double  _profit;           // 当前盈亏金额
      char    _opentag[32];      // 开仓标签
      uint32_t _open_barno;      // 开仓时的调度次数（Bar序号）
    } DetailInfo;
    ```

- **交易信号信息**
  - `SignalMap _sig_map`：信号映射表，记录每个合约的持仓信号
    - typedef `wt_hashmap<std::string, SigInfo>` SignalMap
    ```cpp
    typedef struct _SigInfo
    {
      double    _volume;      // 目标持仓数量
      std::string _usertag;    // 用户标签
      double    _sigprice;     // 信号价格，信号生成时的价格
      double    _desprice;     // 期望成交价格，如果为0则使用当前价格
      bool      _triggered;     // 是否已触发
      uint64_t  _gentime;       // 信号生成时间（纳秒时间戳）
    } SigInfo;
    ```

- **日志流**
  - `std::stringstream _trade_logs`：交易日志流，用于记录交易记录
  - `std::stringstream _close_logs`：平仓日志流，用于记录平仓记录
  - `std::stringstream _fund_logs`：资金日志流，用于记录资金曲线
  - `std::stringstream _sig_logs`：信号日志流，用于记录持仓信号
  - `std::stringstream _pos_logs`：持仓日志流，用于记录持仓变化

- **调度状态**
  - `bool _is_in_schedule`：是否在自动调度中，用于标记是否正在执行定时调度

- **用户数据**
  - `StringHashMap _user_datas`：用户数据映射表，存储用户自定义数据
    - typedef `wt_hashmap<std::string, std::string>` StringHashMap
  - `bool _ud_modified`：用户数据是否已修改，用于判断是否需要保存用户数据

- **资金信息**
  - `StraFundInfo _fund_info`：策略资金信息对象，记录策略的资金状态
    ```cpp
    typedef struct _StraFundInfo
    {
      double  _total_profit;      // 总已平仓盈亏金额
      double  _total_dynprofit;    // 总动态盈亏金额
      double  _total_fees;         // 总手续费金额
    } StraFundInfo;
    ```

- **策略工厂和策略实例**
  - `StraFactInfo _factory`：策略工厂信息对象，管理策略工厂的生命周期
    ```cpp
    typedef struct _StraFactInfo
    {
      std::string   _module_path;      // 策略模块路径
      DllHandle     _module_inst;      // 动态库句柄
      ISelStrategyFact* _fact;         // 策略工厂指针
      FuncCreateSelStraFact _creator;   // 创建工厂函数指针
      FuncDeleteSelStraFact _remover;   // 删除工厂函数指针
    } StraFactInfo;
    ```
  - `SelStrategy* _strategy`：策略实例指针，指向当前执行的策略对象

- **时间信息**
  - `uint32_t _cur_tdate`：当前交易日日期（格式：YYYYMMDD）

- **Tick订阅**
  - `wt_hashset<std::string> _tick_subs`：Tick订阅列表，记录已订阅Tick数据的合约代码集合

### 初始化与配置方法

#### 初始化选股策略工厂 init_sel_factory

### IDataSink 接口实现

#### 处理Tick数据 handle_tick

#### 处理K线闭合事件 handle_bar_close

#### 处理定时调度事件 handle_schedule

#### 处理初始化事件 handle_init

#### 处理交易日开始事件 handle_session_begin

#### 处理交易日结束事件 handle_session_end

#### 处理回测完成事件 handle_replay_done

### ISelStraCtx 接口实现

#### 策略生命周期回调

##### 获取上下文ID id

##### 初始化完成回调 on_init

##### 交易日开始回调 on_session_begin

##### 交易日结束回调 on_session_end

##### Tick数据回调 on_tick

##### K线数据回调 on_bar

##### 定时调度回调 on_schedule

##### 枚举持仓 enum_position

##### Tick数据更新回调 on_tick_updated

##### K线闭合回调 on_bar_close

##### 策略定时调度回调 on_strategy_schedule

#### 交易操作接口

##### 获取持仓数量 stra_get_position

##### 设置目标持仓 stra_set_position

#### 价格与时间信息接口

##### 获取当前价格 stra_get_price

##### 读取当日价格 stra_get_day_price

##### 获取当前交易日日期 stra_get_tdate

##### 获取当前日期 stra_get_date

##### 获取当前时间 stra_get_time

#### 资金信息接口

##### 获取资金数据 stra_get_fund_data

#### 持仓信息接口

##### 获取首次开仓时间 stra_get_first_entertime

##### 获取最后一次开仓时间 stra_get_last_entertime

##### 获取最后一次平仓时间 stra_get_last_exittime

##### 获取最后一次开仓价格 stra_get_last_enterprice

##### 获取最后一次开仓标签 stra_get_last_entertag

##### 获取持仓平均成本价 stra_get_position_avgpx

##### 获取持仓盈亏 stra_get_position_profit

##### 获取指定标签的持仓开仓时间 stra_get_detail_entertime

##### 获取指定标签的持仓成本价 stra_get_detail_cost

##### 获取指定标签的持仓盈亏 stra_get_detail_profit

#### 市场数据接口

##### 获取合约信息 stra_get_comminfo

##### 获取交易时间模板信息 stra_get_sessinfo

##### 获取K线数据切片 stra_get_bars

##### 获取Tick数据切片 stra_get_ticks

##### 获取最新Tick数据 stra_get_last_tick

##### 获取分月合约代码 stra_get_rawcode

##### 订阅Tick数据 stra_sub_ticks

#### 日志接口

##### 记录信息日志 stra_log_info

##### 记录调试日志 stra_log_debug

##### 记录警告日志 stra_log_warn

##### 记录错误日志 stra_log_error

#### 用户数据接口 

##### 保存用户数据 stra_save_user_data

##### 加载用户数据 stra_load_user_data

### 内部辅助方法

#### 输出回测结果文件 dump_outputs

#### 输出策略状态数据 dump_stradata

#### 记录信号日志 log_signal

#### 记录交易日志 log_trade

#### 记录平仓日志 log_close

#### 更新动态盈亏 update_dyn_profit

#### 执行持仓设置 do_set_position

#### 追加持仓信号 append_signal

#### 处理Tick数据 proc_tick

## HftMocker.h/cpp - 高频交易策略回测模拟器
模拟高频交易（HFT）策略在历史数据上的执行过程
```cpp
class HftMocker : public IDataSink, public IHftStraCtx
```

### 架构图
```mermaid
classDiagram
    class IHftStraCtx {
        <<interface>>
        高频策略上下文接口
        为策略提供交易上下文环境
    }
    
    class IDataSink {
        <<interface>>
        数据接收接口
        接收历史数据回放器推送的市场数据
    }
    
    class HftMocker {
        高频策略回测模拟器
        模拟高频交易策略在历史数据上的执行过程
    }
    
    class HisDataReplayer {
        历史数据回放器
        加载和回放历史市场数据
        提供数据查询接口
    }
    
    class HftStrategy {
        <<interface>>
        高频策略接口
        用户实现的策略逻辑
    }
    
    class IHftStrategyFact {
        <<interface>>
        策略工厂接口
        创建和管理策略实例
    }
    
    class StraFactInfo {
        策略工厂信息
        管理动态库加载和策略工厂
    }
    
    IHftStraCtx <|.. HftMocker : 实现
    IDataSink <|.. HftMocker : 实现
    HftMocker --> HisDataReplayer : 使用
    HftMocker --> HftStrategy : 管理
    HftMocker --> StraFactInfo : 包含
    StraFactInfo --> IHftStrategyFact : 包含
    IHftStrategyFact ..> HftStrategy : 创建
```

### 成员
- **核心管理器指针与基础配置**
  - `uint32_t _context_id`：上下文ID
  - `HisDataReplayer* _replayer`：历史数据回放器指针

- **撮合配置**
  - `bool _use_newpx`：是否使用新价格
  - `uint32_t _error_rate`：错误率（用于模拟订单失败）
  - `bool _match_this_tick`：是否在当前tick撮合

- **价格缓存**
  - `PriceMap _price_map`：价格映射表（合约代码->最新价格）
    - typedef `wt_hashmap<std::string, double>` PriceMap

- **策略工厂和策略实例**
  - `StraFactInfo _factory`：策略工厂信息
    ```cpp
    typedef struct _StraFactInfo
    {
      std::string   _module_path;      // 模块路径
      DllHandle     _module_inst;      // 动态库句柄
      IHftStrategyFact* _fact;         // 策略工厂指针
      FuncCreateHftStraFact _creator;  // 创建工厂函数指针
      FuncDeleteHftStraFact _remover;   // 删除工厂函数指针
    } StraFactInfo;
    ```
  - `HftStrategy* _strategy`：HFT策略实例指针

- **任务队列管理**
  - `StdUniqueMutex _mtx`：互斥锁（用于任务队列）
  - `std::queue<Task> _tasks`：任务队列
    - typedef `std::function<void()>` Task
  - `StdRecurMutex _mtx_control`：递归互斥锁（用于控制）

- **订单管理**
  - `Orders _orders`：订单映射表（订单ID->订单信息）
    - typedef `wt_hashmap<uint32_t, OrderInfoPtr>` Orders
    - typedef `std::shared_ptr<OrderInfo>` OrderInfoPtr
    ```cpp
    typedef struct _OrderInfo
    {
      bool    _isBuy;            // 是否买入
      char    _code[32];         // 合约代码
      double  _price;            // 订单价格
      double  _total;            // 总数量
      double  _left;             // 剩余数量
      char    _usertag[32];      // 用户标签
      uint32_t _localid;         // 本地订单ID
      bool    _proced_after_placed;  // 下单后是否处理过
    } OrderInfo;
    ```
  - `StdRecurMutex _mtx_ords`：订单互斥锁

- **合约映射**
  - `CommodityMap* _commodities`：合约映射表指针
    - typedef `WTSHashMap<std::string>` CommodityMap

- **用户数据**
  - `StringHashMap _user_datas`：用户数据映射表
    - typedef `wt_hashmap<std::string, std::string>` StringHashMap
  - `bool _ud_modified`：用户数据是否已修改

- **持仓明细信息**
  - `PositionMap _pos_map`：持仓映射表（合约代码->持仓信息）
    - typedef `wt_hashmap<std::string, PosInfo>` PositionMap
    ```cpp
    typedef struct _PosInfo
    {
      double    _volume;          // 持仓数量
      double    _closeprofit;     // 平仓盈亏
      double    _dynprofit;       // 浮动盈亏
      double    _frozen;          // 冻结数量
      std::vector<DetailInfo> _details;  // 持仓明细列表
    } PosInfo;
    
    typedef struct _DetailInfo
    {
      bool    _long;            // 是否多头
      double  _price;           // 开仓价格
      double  _volume;          // 持仓数量
      uint64_t _opentime;        // 开仓时间
      uint32_t _opentdate;       // 开仓日期
      double  _max_profit;       // 最大盈利
      double  _max_loss;         // 最大亏损
      double  _profit;           // 当前盈亏
      char    _usertag[32];      // 用户标签
    } DetailInfo;
    ```

- **日志流**
  - `std::stringstream _trade_logs`：成交日志流
  - `std::stringstream _close_logs`：平仓日志流
  - `std::stringstream _fund_logs`：资金日志流
  - `std::stringstream _sig_logs`：信号日志流
  - `std::stringstream _pos_logs`：持仓日志流

- **资金信息**
  - `StraFundInfo _fund_info`：策略资金信息
    ```cpp
    typedef struct _StraFundInfo
    {
      double  _total_profit;      // 总盈亏
      double  _total_dynprofit;   // 总浮动盈亏
      double  _total_fees;         // 总手续费
    } StraFundInfo;
    ```

- **异步回测控制**
  - `StdUniqueMutex _mtx_calc`：计算互斥锁
  - `StdCondVariable _cond_calc`：计算条件变量
  - `bool _has_hook`：是否启用钩子（人为控制）
  - `bool _hook_valid`：钩子是否有效（根据异步回测模式确定）
  - `std::atomic<bool> _resumed`：临时变量，用于控制状态（是否已恢复）

- **Tick订阅与缓存**
  - `wt_hashset<std::string> _tick_subs`：tick订阅列表
  - `TickCache* _ticks`：tick缓存指针
    - typedef `WTSHashMap<std::string>` TickCache

### 初始化与配置方法

#### 初始化HFT策略工厂 init_hft_factory

#### 安装钩子（用于异步回测）install_hook

#### 启用/禁用钩子 enable_hook

#### 步进tick（用于异步回测）step_tick

### IDataSink接口实现

#### 处理tick数据 handle_tick

#### 处理订单队列数据 handle_order_queue

#### 处理订单明细数据 handle_order_detail

#### 处理逐笔成交数据 handle_transaction

#### 处理K线收盘事件 handle_bar_close

#### 处理调度事件 handle_schedule

#### 处理初始化事件 handle_init

#### 处理交易时段开始事件 handle_session_begin

#### 处理交易时段结束事件 handle_session_end

#### 处理回放完成事件 handle_replay_done

#### tick数据更新回调 on_tick_updated

#### 订单队列更新回调 on_ordque_updated

#### 订单明细更新回调 on_orddtl_updated

#### 逐笔成交更新回调 on_trans_updated

### IHftStraCtx接口实现

#### 策略生命周期回调

##### 策略tick数据回调 on_tick

##### 策略订单队列回调 on_order_queue

##### 策略订单明细回调 on_order_detail

##### 策略逐笔成交回调 on_transaction

##### 获取策略上下文ID id

##### 策略初始化回调 on_init

##### 策略K线回调 on_bar

##### 策略交易时段开始回调 on_session_begin

##### 策略交易时段结束回调 on_session_end

#### 交易操作接口

##### 撤单（按订单ID）stra_cancel

##### 撤单（按合约和方向）stra_cancel

##### 买入订单 stra_buy

##### 卖出订单 stra_sell

#### 持仓与订单信息接口

##### 获取持仓数量 stra_get_position

##### 获取持仓均价 stra_get_position_avgpx

##### 获取持仓盈亏 stra_get_position_profit

##### 获取未完成订单数量 stra_get_undone

#### 价格与时间信息接口

##### 获取最新价格 stra_get_price

##### 获取当前日期 stra_get_date

##### 获取当前时间 stra_get_time

##### 获取当前秒数 stra_get_secs

#### 市场数据接口

##### 获取合约信息 stra_get_comminfo

##### 获取K线数据 stra_get_bars

##### 获取tick数据 stra_get_ticks

##### 获取订单明细数据 stra_get_order_detail

##### 获取订单队列数据 stra_get_order_queue

##### 获取逐笔成交数据 stra_get_transaction

##### 获取最新tick数据 stra_get_last_tick

##### 获取分月合约代码 stra_get_rawcode

##### 订阅tick数据 stra_sub_ticks

##### 订阅订单队列数据 stra_sub_order_queues

##### 订阅订单明细数据 stra_sub_order_details

##### 订阅逐笔成交数据 stra_sub_transactions

#### 日志接口

##### 记录信息日志 stra_log_info

##### 记录调试日志 stra_log_debug

##### 记录警告日志 stra_log_warn

##### 记录错误日志 stra_log_error

#### 用户数据接口

##### 保存用户数据 stra_save_user_data

##### 加载用户数据 stra_load_user_data

### 策略回调接口

##### 成交回报回调 on_trade

##### 订单回报回调 on_order

##### 通道就绪回调 on_channel_ready

##### 委托回报回调 on_entrust

### 内部辅助方法

##### 提交任务到任务队列 postTask

##### 处理任务队列 procTask

##### 处理订单 procOrder

##### 设置持仓 do_set_position

##### 更新动态盈亏 update_dyn_profit

##### 输出结果到文件 dump_outputs

##### 记录成交日志 log_trade

##### 记录平仓日志 log_close

## UftMocker.h/cpp - UFT极速策略回测模拟器
模拟UFT（Ultra Fast Trading）极速策略在历史数据上的执行过程
```cpp
class UftMocker : public IDataSink, public IUftStraCtx
```

### 架构图
```mermaid
classDiagram
    class IUftStraCtx {
        <<interface>>
        极速策略上下文接口
        为策略提供交易上下文环境
    }
    
    class IDataSink {
        <<interface>>
        数据接收接口
        接收历史数据回放器推送的市场数据
    }
    
    class UftMocker {
        极速策略回测模拟器
        模拟极速交易策略在历史数据上的执行过程
    }
    
    class HisDataReplayer {
        历史数据回放器
        加载和回放历史市场数据
        提供数据查询接口
    }
    
    class UftStrategy {
        <<interface>>
        极速策略接口
        用户实现的策略逻辑
    }
    
    class IUftStrategyFact {
        <<interface>>
        策略工厂接口
        创建和管理策略实例
    }
    
    class StraFactInfo {
        策略工厂信息
        管理动态库加载和策略工厂
    }
    
    IUftStraCtx <|.. UftMocker : 实现
    IDataSink <|.. UftMocker : 实现
    UftMocker --> HisDataReplayer : 使用
    UftMocker --> UftStrategy : 管理
    UftMocker --> StraFactInfo : 包含
    StraFactInfo --> IUftStrategyFact : 包含
    IUftStrategyFact ..> UftStrategy : 创建
```

### 成员
- **核心管理器指针与基础配置**
  - `uint32_t _context_id`：上下文ID，用于标识策略上下文
  - `HisDataReplayer* _replayer`：历史数据回放器指针，用于获取历史数据和合约信息

- **撮合配置**
  - `bool _use_newpx`：是否使用最新价撮合，true表示使用最新价，false表示使用对手价
  - `uint32_t _error_rate`：错误率（万分之一），用于模拟订单被随机撤销的概率
  - `bool _match_this_tick`：是否在当前tick撮合，true表示在tick回调后撮合，false表示在tick回调前撮合

- **价格缓存**
  - `PriceMap _price_map`：价格映射表，缓存每个合约的最新价格
    - typedef `wt_hashmap<std::string, double>` PriceMap

- **策略工厂和策略实例**
  - `StraFactInfo _factory`：策略工厂信息
    ```cpp
    typedef struct _StraFactInfo
    {
      std::string   _module_path;      // 策略模块路径
      DllHandle     _module_inst;      // 动态库句柄
      IUftStrategyFact* _fact;          // 策略工厂指针
      FuncCreateUftStraFact _creator;   // 创建工厂函数指针
      FuncDeleteUftStraFact _remover;   // 删除工厂函数指针
    } StraFactInfo;
    ```
  - `UftStrategy* _strategy`：策略实例指针

- **任务队列管理**
  - `StdUniqueMutex _mtx`：互斥锁，用于保护任务队列
  - `std::queue<Task> _tasks`：任务队列
    - typedef `std::function<void()>` Task
  - `StdRecurMutex _mtx_control`：递归互斥锁，用于控制任务处理

- **订单管理**
  - `Orders _orders`：订单映射表（本地订单ID -> 订单信息）
    - typedef `wt_hashmap<uint32_t, OrderInfo>` Orders
    ```cpp
    typedef struct _OrderInfo
    {
      bool    _isLong;    // 是否做多
      char    _code[32];  // 合约代码
      double  _price;     // 委托价格
      double  _total;     // 总委托数量
      double  _left;      // 剩余数量
      uint32_t _offset;   // 开平标志：0-开仓，1-平仓，2-平今
      uint32_t _localid;  // 本地订单ID
    } OrderInfo;
    ```
  - `StdRecurMutex _mtx_ords`：互斥锁，用于保护订单映射表

- **用户数据**
  - `StringHashMap _user_datas`：用户数据映射表，用于存储策略的自定义数据
    - typedef `wt_hashmap<std::string, std::string>` StringHashMap
  - `bool _ud_modified`：用户数据是否已修改标志

- **持仓明细信息**
  - `PositionMap _pos_map`：持仓映射表（合约代码 -> 持仓信息）
    - typedef `wt_hashmap<std::string, PosInfo>` PositionMap
    ```cpp
    typedef struct _PosInfo
    {
      PosItem  _long;   // 多头持仓
      PosItem  _short;  // 空头持仓
    } PosInfo;
    
    typedef struct _PosItem
    {
      bool    _long;        // 是否做多
      double  _closeprofit; // 已平仓盈亏
      double  _dynprofit;   // 动态盈亏
      double  _prevol;      // 昨仓数量
      double  _newvol;      // 今仓数量
      double  _preavail;    // 昨仓可用数量
      double  _newavail;    // 今仓可用数量
      std::vector<DetailInfo> _details;  // 持仓明细列表
    } PosItem;
    
    typedef struct _DetailInfo
    {
      double  _price;      // 开仓价格
      double  _volume;     // 持仓数量
      uint64_t _opentime;   // 开仓时间（纳秒时间戳）
      uint32_t _opentdate;  // 开仓交易日
      double  _max_profit;  // 最大盈利金额
      double  _max_loss;    // 最大亏损金额
      double  _profit;      // 当前盈亏金额
    } DetailInfo;
    ```

- **日志流**
  - `std::stringstream _trade_logs`：交易日志流，用于记录交易记录
  - `std::stringstream _close_logs`：平仓日志流，用于记录平仓记录
  - `std::stringstream _fund_logs`：资金日志流，用于记录资金曲线
  - `std::stringstream _pos_logs`：持仓日志流，用于记录持仓记录

- **资金信息**
  - `StraFundInfo _fund_info`：策略资金信息
    ```cpp
    typedef struct _StraFundInfo
    {
      double  _total_profit;      // 总已平仓盈亏
      double  _total_dynprofit;   // 总动态盈亏
      double  _total_fees;         // 总手续费
    } StraFundInfo;
    ```

- **Tick订阅**
  - `wt_hashset<std::string> _tick_subs`：Tick数据订阅列表，存储已订阅Tick数据的合约代码集合

### 初始化与配置方法

#### 初始化UFT策略工厂 init_uft_factory

### IDataSink 接口实现

#### 处理Tick数据 handle_tick

#### 处理委托队列数据 handle_order_queue

#### 处理委托明细数据 handle_order_detail

#### 处理逐笔成交数据 handle_transaction

#### 处理K线闭合事件 handle_bar_close

#### 处理定时调度事件 handle_schedule

#### 处理初始化事件 handle_init

#### 处理交易日开始事件 handle_session_begin

#### 处理交易日结束事件 handle_session_end

#### 处理回测完成事件 handle_replay_done

#### Tick数据更新回调 on_tick_updated

#### 委托队列数据更新回调 on_ordque_updated

#### 委托明细数据更新回调 on_orddtl_updated

#### 逐笔成交数据更新回调 on_trans_updated

### IUftStraCtx 接口实现

#### 策略生命周期回调

##### Tick数据回调 on_tick

##### 委托队列数据回调 on_order_queue

##### 委托明细数据回调 on_order_detail

##### 逐笔成交数据回调 on_transaction

##### 获取上下文ID id

##### 初始化完成回调 on_init

##### K线数据回调 on_bar

##### 交易日开始回调 on_session_begin

##### 交易日结束回调 on_session_end

#### 交易操作接口

##### 撤销指定订单 stra_cancel

##### 撤销指定合约的所有订单 stra_cancel_all

##### 买入（智能处理平空和开多）stra_buy

##### 卖出（智能处理平多和开空）stra_sell

##### 开多 stra_enter_long

##### 开空 stra_enter_short

##### 平多 stra_exit_long

##### 平空 stra_exit_short

#### 持仓与订单信息接口

##### 获取持仓 stra_get_position

##### 获取本地持仓（净头寸）stra_get_local_position

##### 枚举持仓 stra_enum_position

##### 获取未成交数量 stra_get_undone

#### 价格与时间信息接口

##### 获取当前价格 stra_get_price

##### 获取当前日期 stra_get_date

##### 获取当前时间 stra_get_time

##### 获取当前秒数 stra_get_secs

#### 市场数据接口

##### 获取合约信息 stra_get_comminfo

##### 获取K线数据切片 stra_get_bars

##### 获取Tick数据切片 stra_get_ticks

##### 获取委托明细数据切片 stra_get_order_detail

##### 获取委托队列数据切片 stra_get_order_queue

##### 获取逐笔成交数据切片 stra_get_transaction

##### 获取最新Tick数据 stra_get_last_tick

##### 订阅Tick数据 stra_sub_ticks

##### 订阅委托队列数据 stra_sub_order_queues

##### 订阅委托明细数据 stra_sub_order_details

##### 订阅逐笔成交数据 stra_sub_transactions

#### 日志接口

##### 记录信息日志 stra_log_info

##### 记录调试日志 stra_log_debug

##### 记录错误日志 stra_log_error

### 策略回调接口

#### 成交回调 on_trade

#### 订单状态回调 on_order

#### 通道就绪回调 on_channel_ready

#### 委托回调 on_entrust

### 内部辅助方法

#### 提交任务到任务队列 postTask

#### 处理任务队列 procTask

#### 处理订单撮合 procOrder

#### 更新持仓 update_position

#### 更新动态盈亏 update_dyn_profit

#### 输出回测结果文件 dump_outputs

#### 记录交易日志 log_trade

#### 记录平仓日志 log_close

## ExecMocker.h/cpp - 执行器模拟器
模拟执行器在历史数据上的订单执行过程（专注于订单执行，不涉及策略逻辑）
```cpp
class ExecMocker : public ExecuteContext, public IDataSink, public IMatchSink
```

### 架构图
```mermaid
classDiagram
    class ExecuteContext {
        <<interface>>
        执行器上下文接口
        为执行器提供执行上下文环境
    }
    
    class IDataSink {
        <<interface>>
        数据接收接口
        接收历史数据回放器推送的市场数据
    }
    
    class IMatchSink {
        <<interface>>
        撮合回调接口
        接收撮合引擎的订单状态变化通知
    }
    
    class ExecMocker {
        执行器模拟器
        模拟执行器在历史数据上的订单执行过程
    }
    
    class HisDataReplayer {
        历史数据回放器
        加载和回放历史市场数据
        提供数据查询接口
    }
    
    class MatchEngine {
        撮合引擎
        模拟订单撮合过程
    }
    
    class ExecuteUnit {
        <<interface>>
        执行器单元接口
        用户实现的执行器逻辑
    }
    
    class IExecuterFact {
        <<interface>>
        执行器工厂接口
        创建和管理执行器实例
    }
    
    class ExecFactInfo {
        执行器工厂信息
        管理动态库加载和执行器工厂
    }
    
    ExecuteContext <|.. ExecMocker : 实现
    IDataSink <|.. ExecMocker : 实现
    IMatchSink <|.. ExecMocker : 实现
    ExecMocker --> HisDataReplayer : 使用
    ExecMocker --> MatchEngine : 使用
    ExecMocker --> ExecuteUnit : 管理
    ExecMocker --> ExecFactInfo : 包含
    ExecFactInfo --> IExecuterFact : 包含
    IExecuterFact ..> ExecuteUnit : 创建
    MatchEngine --> IMatchSink : 回调
```

### 成员
- **核心管理器指针与基础配置**
  - `HisDataReplayer* _replayer`：历史数据回放器指针

- **执行器工厂和实例**
  - `ExecFactInfo _factory`：执行器工厂信息
    ```cpp
    typedef struct _ExecFactInfo
    {
      std::string   _module_path;      // 模块路径
      DllHandle     _module_inst;      // 动态库句柄
      IExecuterFact* _fact;            // 执行器工厂指针
      FuncCreateExeFact _creator;      // 创建工厂函数指针
      FuncDeleteExeFact _remover;      // 删除工厂函数指针
    } ExecFactInfo;
    ```
  - `ExecuteUnit* _exec_unit`：执行器单元指针

- **执行配置**
  - `std::string _code`：合约代码
  - `std::string _period`：周期
  - `double _volunit`：数量单位
  - `int32_t _volmode`：数量模式：0-反复正负，-1-一直卖，+1-一直买

- **目标与持仓**
  - `double _target`：目标仓位
  - `double _position`：当前持仓
  - `double _undone`：未完成订单数量

- **市场数据**
  - `WTSTickData* _last_tick`：最新tick数据
  - `double _sig_px`：信号价格
  - `uint64_t _sig_time`：信号时间

- **统计信息**
  - `std::stringstream _trade_logs`：成交日志流
  - `uint32_t _ord_cnt`：订单数量统计
  - `double _ord_qty`：订单数量统计
  - `uint32_t _cacl_cnt`：撤单数量统计
  - `double _cacl_qty`：撤单数量统计
  - `uint32_t _sig_cnt`：信号数量统计

- **标识信息**
  - `std::string _id`：执行器ID

- **撮合引擎**
  - `MatchEngine _matcher`：撮合引擎

### 初始化与配置方法

#### 初始化执行器模拟器 init

### IMatchSink 接口实现

#### 处理成交回报 handle_trade

#### 处理订单回报 handle_order

#### 处理委托回报 handle_entrust

### IDataSink 接口实现

#### 处理tick数据 handle_tick

#### 处理调度事件 handle_schedule

#### 处理初始化事件 handle_init

#### 处理K线收盘事件 handle_bar_close

#### 处理交易时段开始事件 handle_session_begin

#### 处理交易时段结束事件 handle_session_end

#### 处理回放完成事件 handle_replay_done

### ExecuteContext 接口实现 

#### 数据查询接口

##### 获取tick数据切片 getTicks

##### 获取最新tick数据 grabLastTick

##### 获取持仓数量 getPosition

##### 获取订单映射表 getOrders

##### 获取未完成订单数量 getUndoneQty

#### 交易操作接口

##### 买入订单 buy

##### 卖出订单 sell

##### 撤单（按订单ID）cancel

##### 撤单（按合约和方向）cancel

#### 信息查询接口

##### 写日志 writeLog

##### 获取合约信息 getCommodityInfo

##### 获取交易时段信息 getSessionInfo

##### 获取当前时间 getCurTime

# 辅助层

## MatchEngine.h/cpp - 撮合引擎
模拟订单撮合过程，包括订单管理、撮合逻辑、限价订单簿维护

### 撮合引擎回调接口 IMatchSink
用于接收撮合引擎的订单状态变化通知··
```cpp
class IMatchSink
```

#### 方法

##### 成交回报回调 handle_trade

##### 订单回报回调 handle_order

##### 委托回报回调 handle_entrust

### 撮合引擎类 MatchEngine`
```cpp
class MatchEngine
```

#### 成员
- **核心回调接口与配置**
  - `IMatchSink* _sink`：撮合回调接口指针，接收撮合引擎的订单状态变化通知（成交、订单、委托回报）
  - `double _cancelrate`：撤单率（0-1），模拟订单被随机撤销的概率
  - `WTSTickCache* _tick_cache`：Tick缓存指针，指向Tick缓存，用于获取最新Tick数据

- **订单管理**
  - `Orders _orders`：订单映射表，存储所有订单信息，以订单ID为键
    - typedef wt_hashmap<uint32_t, `OrderInfo`> Orders
        ```cpp
        typedef struct _OrderInfo
        {
        char        _code[32];      // 合约代码
        bool        _buy;           // 是否买入
        double      _qty;           // 订单数量
        double      _left;          // 剩余数量
        double      _traded;        // 已成交数量
        double      _limit;         // 限价
        double      _price;         // 订单价格
        uint32_t    _state;         // 订单状态（0-待激活，1-已激活，9-待撤单，99-已撤单）
        uint64_t    _time;          // 订单时间
        double      _queue;         // 排队位置
        bool        _positive;      // 是否主动订单（对手价）
        } OrderInfo;
        ```

- **限价订单簿管理**
  - `LmtOrdBooks _lmt_ord_books`：限价订单簿映射表，维护每个合约的限价订单簿，记录价格档位和数量
    - typedef wt_hashmap<std::string, `LmtOrdBook`> LmtOrdBooks
    - typedef std::map<uint32_t, double> `LOBItems`：限价订单簿项（价格（整数）-> 数量）
        ```cpp
        typedef struct _LmtOrdBook
        {
        LOBItems    _items;         // 订单簿项（价格 -> 数量）
        uint32_t    _cur_px;        // 当前价格（整数）
        uint32_t    _ask_px;        // 卖一价（整数）
        uint32_t    _bid_px;        // 买一价（整数）
        } LmtOrdBook;
        ```

#### 核心属性

##### 初始化撮合引擎 init

##### 注册回调接口 regisSink

#### 订单管理方法

##### 买入订单 buy

##### 卖出订单 sell

##### 撤销订单（按订单ID）cancel

##### 撤销订单（按合约代码和方向）cancel

##### 清空所有订单 clear

#### 数据处理方法

##### 处理Tick数据 handle_tick

#### 内部辅助方法

##### 激活订单 fire_orders

##### 撮合订单 match_orders

##### 更新限价订单簿 update_lob

##### 获取最新Tick数据 grab_last_tick

## EventNotifier.h/cpp - 事件通知器
在回测过程中向外发送事件和数据通知
```cpp
class EventNotifier
```

### 成员
- **消息队列配置与连接**
  - `std::string m_strURL`：消息队列URL地址，指定消息队列服务器的位置
  - `uint32_t _mq_sid`：消息队列服务器ID，标识已创建的消息队列服务器实例

- **消息队列服务函数指针**
  - `FuncCreateMQServer _creator`：创建MQ服务器的函数指针，动态库中创建消息队列服务器的函数指针
    - typedef unsigned long\(\*`FuncCreateMQServer`\)\(const char* url, bool bServer\)
  - `FuncDestroyMQServer _remover`：销毁MQ服务器的函数指针，动态库中销毁消息队列服务器的函数指针
    - typedef void\(\*`FuncDestroyMQServer`\)\(unsigned long sid\)
  - `FundPublishMessage _publisher`：发布消息的函数指针，动态库中发布消息的函数指针
    - typedef void\(\*`FundPublishMessage`\)\(unsigned long sid, const char* topic, const char* data, unsigned long dataLen\)
  - `FuncRegCallbacks _register`：注册回调函数的函数指针，动态库中注册回调函数的函数指针
    - typedef void\(\*`FuncRegCallbacks`\)\(FuncLogCallback logCb\)

### 初始化与配置

#### 初始化事件通知器 init

### 事件通知方法

#### 通知事件 notifyEvent

#### 通知数据 notifyData

#### 通知资金信息 notifyFund

# 工具层

## WtHelper.h/cpp
提供通用辅助功能
```cpp
class WtHelper
```

### 成员
- `static std::string _inst_dir`：实例所在目录，存储实例目录路径
- `static std::string _out_dir`：输出目录，存储回测结果输出目录路径，默认为"./outputs_bt/"

### 方法

#### 获取当前工作目录 getCWD

#### 获取输出目录 getOutputDir

#### 获取实例目录 getInstDir

#### 设置实例目录 setInstDir

#### 设置输出目录 setOutputDir